# DramaFlow Backend (Colab) — Recap + Link ဆွဲ + Creator Studio
GitHub branch မလိုဘူး၊ backend ကုဒ်အားလုံး ဒီ notebook ထဲမှာ ပါပြီးသား။ (Video Recap / Scene Recap "Auto" + Edge TTS + Self-test + **ဗီဒီယို link (YouTube/TikTok/Red Note/Bilibili) ဆွဲပေးတာ** + **Creator Studio** — download၊ ဖြတ်၊ စာတန်း ဘာသာပြန်၊ CC/Hardsub၊ ZIP၊ YouTube တင်)

**သုံးပုံ**
1. Colab (colab.research.google.com) > File > Upload notebook > ဒီဖိုင်ကို ရွေး
2. Cell တွေကို အစီအစဉ်အတိုင်း **တစ်ခုချင်း run** (Cell 1 → ၂ → … → နောက်ဆုံး server Cell)
3. နောက်ဆုံး Cell ကထုတ်ပေးတဲ့ `https://….trycloudflare.com#TOKEN` တစ်ကြောင်းလုံးကို DramaFlow ရဲ့ Backend ကွက်မှာ ထည့်၊ "သိမ်း + စမ်းချိတ်" နှိပ်
4. Colab ရပ်ရင်/ပိတ်ရင် URL ပြောင်းမယ် — နောက်ဆုံး Cell ကို ပြန် run ပြီး အသစ်ယူ။ runtime ပြန်စထားရင် Cell တွေ အကုန်ပြန် run။ **Studio ထဲက ဖိုင်တွေက Colab runtime နဲ့အတူ ပျောက်မယ်** — လိုတာကို ဖုန်းထဲ အရင်ဆွဲယူထားပါ။

**ဗီဒီယို link:** ဗီဒီယိုနဲ့အတူ စာတန်း (.srt) ပါရင် အလိုလို ယူပေးတယ်။ YouTube က Colab ကို တားရင် **Cell 5b** မှာ cookies.txt တင်ပါ။
**Creator Studio:** ကိုယ်ပိုင် / တင်ခွင့်ရှိတဲ့ ဗီဒီယိုအတွက်ပဲ။ YouTube တင်ချင်ရင် **Cell 5c** မှာ `client_secret.json` တင်ပါ (Google Cloud > YouTube Data API v3 > OAuth client "Desktop app")။ GPU runtime ရွေးရင် Hardsub ပိုမြန်တယ်။

In [ ]:
# Cell 1: package တွေ + cloudflared + aria2 + မြန်မာဖောင့် တပ်ဆင် (တစ်ကြိမ်၊ ၂ မိနစ်လောက်)
!pip -q install fastapi uvicorn edge-tts google-genai imageio-ffmpeg yt-dlp deep-translator google-api-python-client google-auth-oauthlib google-auth-httplib2 faster-whisper
!apt-get -qq install -y aria2 fonts-sil-padauk > /dev/null 2>&1 || echo "aria2/ဖောင့် မတပ်နိုင်ဘူး — download က 1 လိုင်းနဲ့ပဲ၊ Hardsub က မူရင်းဖောင့်နဲ့ ထွက်မယ်"
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!mkdir -p /content/dramaflow_backend
print("ok")

In [ ]:
%%writefile /content/dramaflow_backend/main.py
"""DramaFlow Recap Backend (for Altare, Colab or any computer that runs Python).

Start:  uvicorn main:app --host 0.0.0.0 --port $PORT
Set the environment variable APP_TOKEN to a long secret of your own. Every call
except /health needs it in the header x-app-token. Without it a random token is
made at start and printed in the log.

What it does for the app (Video Recap and Scene Recap):
  POST /recap/jobs             start a job, answers at once with a job id
  GET  /recap/jobs/<id>        progress, then the script parts (and voice files)
  GET  /recap/jobs/<id>/audio/<n>.wav   the voice of part n (24 kHz mono WAV)
  POST /ingest/jobs            a video link (YouTube, TikTok, Red Note ...): download it
                               here, put it into the user's own Gemini account, answer
                               with the Gemini file address and the length
  GET  /ingest/jobs/<id>       progress, then {file_uri, mime, duration_ms, title}
  Creator Studio (your own files and videos you may use; the library lives on this machine):
  GET  /studio/files           the library          GET /studio/files/<name>   one file to the phone
  DELETE /studio/files/<name>  POST /studio/upload  a small .srt / picture from the phone
  POST /studio/jobs            download | split | translate | subtitle | zip | youtube | transcribe (Whisper) | render
  GET  /studio/jobs/<id>       progress (speed, ETA), result files / links
  POST /studio/jobs/<id>/cancel | pause | resume
  GET/POST /studio/youtube/... status, auth-link, auth-complete, unlink
The Gemini key arrives with the request, is used for the job and is not stored.
"""
import asyncio
import base64
import json
import os
import re
import secrets
import shutil
import subprocess
import tempfile
import threading
import time
import wave
from pathlib import Path
from typing import List, Optional

import edge_tts
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, Response
from google import genai
from google.genai import types
from pydantic import BaseModel

from ingest_core import (MAX_BYTES, MAX_SECONDS, SUB_LANGS, SUB_SUFFIXES, check_url, classify, clean_srt, cookie_note,
                         cookies_problem, cookies_source as _cookies_source, ingest_hint, parse_ffmpeg_duration, pick_subtitle, run_ingest)
from recap_core import fit_tempo, gemini_hint, mask_key, run_recap
import studio_core as sc
import studio_ops as so
from tts_text import normalize_for_tts

app = FastAPI(title="DramaFlow Recap Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

TOKEN = os.environ.get("APP_TOKEN") or secrets.token_urlsafe(18)
if not os.environ.get("APP_TOKEN"):
    print("APP_TOKEN not set. A temporary token for this run:", TOKEN, flush=True)
OUT_DIR = Path(tempfile.gettempdir()) / "dramaflow_recap"
OUT_DIR.mkdir(parents=True, exist_ok=True)
JOBS: dict = {}
LOCK = threading.Lock()
KEEP_SECONDS = 2 * 3600
DEFAULT_VOICE = "my-MM-ThihaNeural"


class Group(BaseModel):
    first: int
    last: int
    prompt: str
    jpeg_b64: str = ""  # Scene Recap: the numbered contact sheet
    start_ms: int = 0  # Video Recap: look only at this time range
    end_ms: int = 0
    part_ms: List[int] = []  # how long each part's video is (first..last), to fit the voice


class RecapRequest(BaseModel):
    mode: str = "scenes"  # "scenes" (pictures + dialogue) or "video" (a file already in Gemini)
    gemini_key: str
    gemini_model: str = "gemini-3.8-flash"
    gemini_fallback: str = ""  # optional second model, tried when the first is gone (404) or over its quota
    voice: str = ""  # empty = script only, no voice
    file_uri: str = ""  # Gemini Files API address (video mode)
    file_mime: str = "video/mp4"
    groups: List[Group]


def _auth(token: Optional[str]):
    if not token or not secrets.compare_digest(token, TOKEN):
        raise HTTPException(status_code=401, detail="Bad or missing x-app-token")


def _set(job_id: str, **kw):
    with LOCK:
        JOBS[job_id].update(kw)


def _ffmpeg() -> str:
    try:
        import imageio_ffmpeg

        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return shutil.which("ffmpeg") or "ffmpeg"


FALLBACK_WHEN = ("404", "NOT_FOUND", "429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")


def _gemini_fn(req: RecapRequest):
    client = genai.Client(api_key=req.gemini_key)

    def ask(model: str, contents: list) -> str:
        for attempt in range(3):
            try:
                r = client.models.generate_content(model=model, contents=contents, config=types.GenerateContentConfig(temperature=0.5))
                return r.text or ""
            except Exception as e:
                msg = str(e)
                busy = any(k in msg for k in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE"))
                if busy and attempt < 2:
                    time.sleep(25 * (attempt + 1))
                    continue
                raise RuntimeError(mask_key(msg, req.gemini_key))
        return ""

    def call(g: dict) -> str:
        contents: list = []
        if g.get("jpeg_b64"):
            contents.append(types.Part.from_bytes(data=base64.b64decode(g["jpeg_b64"]), mime_type="image/jpeg"))
        if req.mode == "video":
            if not req.file_uri:
                raise ValueError("video file_uri မပါဘူး")
            contents.append(
                types.Part(
                    file_data=types.FileData(file_uri=req.file_uri, mime_type=req.file_mime),
                    video_metadata=types.VideoMetadata(
                        start_offset=f"{max(0, int(g.get('start_ms', 0)) // 1000)}s",
                        end_offset=f"{max(1, -(-int(g.get('end_ms', 0)) // 1000))}s",
                    ),
                )
            )
        contents.append(g["prompt"])
        try:
            return ask(req.gemini_model, contents)
        except RuntimeError as e:
            fb = req.gemini_fallback.strip()
            if fb and fb != req.gemini_model and any(k in str(e) for k in FALLBACK_WHEN):
                print(f"model {req.gemini_model} failed, trying {fb}", flush=True)
                return ask(fb, contents)
            raise

    return call


def _speak(text: str, voice: str, mp3: Path, rate: str = "+0%"):
    """Edge TTS to an mp3 file, three tries."""
    for attempt in range(3):
        try:
            asyncio.run(edge_tts.Communicate(text, voice, rate=rate).save(str(mp3)))
            return
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2 * (attempt + 1))


def _to_wav(src: Path, dst: Path, tempo: float = 1.0):
    """24 kHz mono WAV; tempo above 1.0 speeds the voice up (pitch is kept)."""
    cmd = [_ffmpeg(), "-y", "-i", str(src), "-ar", "24000", "-ac", "1"]
    if tempo > 1.001:
        cmd += ["-filter:a", f"atempo={tempo}"]
    cmd += ["-c:a", "pcm_s16le", str(dst)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0 or not dst.exists():
        raise RuntimeError("ffmpeg: " + (p.stderr or "")[-200:])


def _wav_ms(wav: Path) -> int:
    with wave.open(str(wav), "rb") as w:
        return int(w.getnframes() * 1000 / w.getframerate())


def _tts_fn(job_dir: Path, voice: str):
    use = voice if voice.startswith("my-MM-") else DEFAULT_VOICE

    def make(n: int, text: str, target_ms: int = 0):
        mp3, wav = job_dir / f"p{n}.mp3", job_dir / f"p{n}.wav"
        try:
            _speak(normalize_for_tts(text), use, mp3)
            _to_wav(mp3, wav)
            ms = _wav_ms(wav)
            tempo = fit_tempo(ms, target_ms)
            if tempo > 1.0:  # a little too long: speed it up, never more than 15%
                fast = job_dir / f"p{n}_fast.wav"
                _to_wav(mp3, fast, tempo)
                fast.replace(wav)
                ms = _wav_ms(wav)
        finally:
            mp3.unlink(missing_ok=True)
        return ms, wav.name

    return make


def clean_old():
    now = time.time()
    with LOCK:
        for jid, j in list(JOBS.items()):
            if j["status"] in ("done", "error") and now - j["created"] > KEEP_SECONDS:
                shutil.rmtree(OUT_DIR / jid, ignore_errors=True)
                del JOBS[jid]


def _save_job(job_id: str):
    """The finished job (text, warnings, audio addresses) is kept next to its
    audio files, so the app can fetch it again after the web process restarted."""
    try:
        with LOCK:
            data = dict(JOBS[job_id])
        (OUT_DIR / job_id / "job.json").write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    except Exception:
        pass


def _load_job(job_id: str):
    if not re.fullmatch(r"recap_\d+_[0-9a-f]{4}", job_id):
        return None
    f = OUT_DIR / job_id / "job.json"
    try:
        data = json.loads(f.read_text(encoding="utf-8"))
    except Exception:
        return None
    if data.get("status") != "done" or time.time() - float(data.get("created", 0)) > KEEP_SECONDS:
        return None
    with LOCK:
        JOBS[job_id] = data
    return data


def run_job(job_id: str, req: RecapRequest):
    job_dir = OUT_DIR / job_id
    try:
        job_dir.mkdir(parents=True, exist_ok=True)
        _set(job_id, status="running", stage="စတင်နေတယ်", progress=2)
        result = run_recap(
            [g.model_dump() for g in req.groups],
            _gemini_fn(req),
            _tts_fn(job_dir, req.voice) if req.voice else None,
            lambda stage, pct: _set(job_id, stage=stage, progress=pct),
        )
        for p in result["parts"]:
            if p["audio_file"]:
                p["audio_url"] = f"/recap/jobs/{job_id}/audio/{p['n']}.wav"
            p.pop("audio_file")
        _set(job_id, status="done", stage="ပြီးပြီ", progress=100, parts=result["parts"], warnings=result["warnings"])
        _save_job(job_id)
    except Exception as e:  # shown in the app, with the key hidden
        _set(job_id, status="error", stage="မအောင်မြင်ဘူး", error=mask_key(str(e), req.gemini_key)[:400])
        shutil.rmtree(job_dir, ignore_errors=True)


@app.get("/health")
def health():
    return {"status": "ok", "service": "DramaFlow Recap Backend", "features": ["recap", "tts", "ingest", "studio"]}


@app.post("/recap/jobs")
def create_job(req: RecapRequest, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if not req.groups:
        raise HTTPException(status_code=400, detail="groups မပါဘူး")
    clean_old()
    with LOCK:
        if any(j["status"] in ("queued", "running") for j in JOBS.values()):
            raise HTTPException(status_code=409, detail="တခြား job တစ်ခု လုပ်နေတုန်းပါ")
        job_id = f"recap_{int(time.time())}_{secrets.token_hex(2)}"
        JOBS[job_id] = {"status": "queued", "stage": "စောင့်နေတယ်", "progress": 0, "created": time.time(),
                        "error": "", "parts": [], "warnings": []}
    threading.Thread(target=run_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id}


@app.get("/recap/jobs/{job_id}")
def job_status(job_id: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    with LOCK:
        j = JOBS.get(job_id)
    if j is None:
        j = _load_job(job_id)
    if j is None:
        raise HTTPException(status_code=404, detail="Job မတွေ့ပါ")
    with LOCK:
        return {k: v for k, v in j.items() if k != "created"}


@app.get("/recap/jobs/{job_id}/audio/{n}.wav")
def audio(job_id: str, n: int, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if not re.fullmatch(r"recap_\d+_[0-9a-f]{4}", job_id):
        raise HTTPException(status_code=404, detail="Audio not found")
    p = OUT_DIR / job_id / f"p{int(n)}.wav"
    if not p.exists():
        raise HTTPException(status_code=404, detail="Audio not found")
    return FileResponse(p, media_type="audio/wav", filename=p.name)


# ---- link -> video in the user's own Gemini account -------------------------
class IngestRequest(BaseModel):
    url: str
    gemini_key: str
    max_height: int = 720  # Gemini looks at about 1 frame a second, so 720p is plenty


def _state_name(f) -> str:
    st = getattr(f, "state", None)
    return str(getattr(st, "name", st) or "")


def _ytdlp_download(url: str, job_dir: Path, max_height: int, on_pct):
    """yt-dlp is imported here so the backend still starts (and recaps work)
    when it is not installed; the error then says what to install."""
    try:
        import yt_dlp
        from yt_dlp.utils import match_filter_func
    except Exception:
        raise RuntimeError("yt-dlp မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ)")
    h = max(240, min(1080, int(max_height)))

    def hook(d):
        if d.get("status") == "downloading":
            tot = d.get("total_bytes") or d.get("total_bytes_estimate") or 0
            if tot:
                on_pct(int(100 * (d.get("downloaded_bytes") or 0) / tot))

    cookies_text = ""
    cookie_file = None
    src = _cookies_source()
    if src is not None:
        cookies_text = src.read_text(encoding="utf-8", errors="ignore")
        bad = cookies_problem(cookies_text)
        if bad:
            raise ValueError(bad)
        cookie_file = job_dir / "cookies.txt"  # a private copy (yt-dlp writes to its cookie file); deleted with the job
        cookie_file.write_text(cookies_text, encoding="utf-8")
        try:
            os.chmod(cookie_file, 0o600)
        except OSError:
            pass
        note = cookie_note(cookies_text, classify(url))
        if note:
            print("cookies:", note, flush=True)

    opts = {
        "outtmpl": str(job_dir / "v.%(ext)s"),
        "format": f"bv*[height<={h}][ext=mp4]+ba[ext=m4a]/bv*[height<={h}]+ba/b[height<={h}]/b",
        "merge_output_format": "mp4",
        "noplaylist": True,
        "max_filesize": MAX_BYTES,
        "match_filter": match_filter_func(f"!is_live & duration <=? {MAX_SECONDS}"),
        "quiet": True,
        "no_warnings": True,
        "socket_timeout": 30,
        "retries": 3,
        "ffmpeg_location": _ffmpeg(),
        "progress_hooks": [hook],
        # the uploader's own subtitles, as .srt (a failure here does not stop the video)
        "writesubtitles": True,
        "subtitleslangs": list(SUB_LANGS),
        "subtitlesformat": "srt/best",
        "postprocessors": [{"key": "FFmpegSubtitlesConvertor", "format": "srt", "when": "before_dl"}],
    }
    if cookie_file is not None:
        opts["cookiefile"] = str(cookie_file)
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
    files = [f for f in job_dir.glob("v.*") if f.suffix.lower() not in (".part", ".ytdl") + tuple(SUB_SUFFIXES)]
    if not files:
        raise RuntimeError("ဗီဒီယို မရဘူး (၃ နာရီထက် ရှည်တာ / live / ၂ GB ထက်ကြီးတာ ဖြစ်နိုင်တယ်)")
    path = max(files, key=lambda f: f.stat().st_size)
    if isinstance(info, dict) and info.get("entries"):
        info = (info["entries"] or [{}])[0] or {}
    ms = int(float((info or {}).get("duration") or 0) * 1000)
    if not ms:
        try:
            p = subprocess.run([_ffmpeg(), "-i", str(path)], capture_output=True, text=True, timeout=60)
            ms = parse_ffmpeg_duration(p.stderr)
        except Exception:
            ms = 0
    srt, srt_lang = "", ""
    sub_name, srt_lang = pick_subtitle([f.name for f in job_dir.glob("v.*.srt")])
    if sub_name:
        try:
            srt = clean_srt((job_dir / sub_name).read_text(encoding="utf-8", errors="ignore"))
        except OSError:
            srt = ""
        if not srt:
            srt_lang = ""
    return {"path": str(path), "title": (info or {}).get("title") or "", "duration_ms": ms,
            "srt": srt, "srt_lang": srt_lang, "cookies": cookie_file is not None}


def _gemini_upload(key: str, path: str, mime: str):
    client = genai.Client(api_key=key)
    f = client.files.upload(file=path, config=types.UploadFileConfig(mime_type=mime, display_name="dramaflow_ingest"))
    waited = 0
    while _state_name(f) == "PROCESSING":
        if waited > 1200:
            raise RuntimeError("Gemini က ဗီဒီယိုကို မပြီးသေးဘူး (၂၀ မိနစ်ကျော်)")
        time.sleep(5)
        waited += 5
        f = client.files.get(name=f.name)
    if _state_name(f) != "ACTIVE":
        raise RuntimeError("Gemini က ဗီဒီယိုကို လက်မခံဘူး (%s)" % (_state_name(f) or "?"))
    return f.uri, mime


def run_ingest_job(job_id: str, req: IngestRequest):
    job_dir = OUT_DIR / job_id
    done = {}
    try:
        job_dir.mkdir(parents=True, exist_ok=True)
        _set(job_id, status="running", stage="စတင်နေတယ်", progress=2)
        done = run_ingest(
            req.url.strip(),
            lambda url, on_pct: _ytdlp_download(url, job_dir, req.max_height, on_pct),
            lambda path, mime: _gemini_upload(req.gemini_key, path, mime),
            lambda stage, pct: _set(job_id, stage=stage, progress=min(pct, 99)),
        )
        err = ""
    except Exception as e:  # shown in the app, with the key hidden
        msg = mask_key(str(e), req.gemini_key)
        hint = ingest_hint(msg, classify(req.url), has_cookies=_cookies_source() is not None)
        err = ((hint + " · ") if hint else "") + msg[:300]
    shutil.rmtree(job_dir, ignore_errors=True)  # the downloaded copy is never kept
    if err:
        _set(job_id, status="error", stage="မအောင်မြင်ဘူး", error=err)
    else:
        _set(job_id, status="done", stage="ပြီးပြီ", progress=100, info=done)


@app.post("/ingest/jobs")
def create_ingest(req: IngestRequest, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if not req.gemini_key.strip():
        raise HTTPException(status_code=400, detail="Gemini key မပါဘူး")
    ok, why = check_url(req.url)
    if not ok:
        raise HTTPException(status_code=400, detail=why)
    clean_old()
    with LOCK:
        if any(j["status"] in ("queued", "running") for j in JOBS.values()):
            raise HTTPException(status_code=409, detail="တခြား job တစ်ခု လုပ်နေတုန်းပါ")
        job_id = f"ingest_{int(time.time())}_{secrets.token_hex(2)}"
        JOBS[job_id] = {"status": "queued", "stage": "စောင့်နေတယ်", "progress": 0, "created": time.time(),
                        "error": "", "info": {}}
    threading.Thread(target=run_ingest_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id}


@app.get("/ingest/jobs/{job_id}")
def ingest_status(job_id: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    with LOCK:
        j = JOBS.get(job_id) if job_id.startswith("ingest_") else None
        if j is None:
            raise HTTPException(status_code=404, detail="Job မတွေ့ပါ")
        return {k: v for k, v in j.items() if k != "created"}


class TtsRequest(BaseModel):
    text: str
    voice: str = DEFAULT_VOICE
    rate: str = "+0%"


@app.post("/tts")
def tts_once(req: TtsRequest, x_app_token: Optional[str] = Header(default=None)):
    """One piece of text -> a WAV (24 kHz mono), for the app's Audio stage."""
    _auth(x_app_token)
    text = req.text.strip()
    if not text or len(text) > 6000:
        raise HTTPException(status_code=400, detail="စာသား မရှိဘူး/ရှည်လွန်းတယ် (၆၀၀၀ လုံးအထိ)")
    rate = req.rate if re.fullmatch(r"[+-]\d{1,3}%", req.rate) else "+0%"
    voice = req.voice if req.voice.startswith("my-MM-") else DEFAULT_VOICE
    d = OUT_DIR / f"tts_{secrets.token_hex(4)}"
    try:
        d.mkdir(parents=True, exist_ok=True)
        mp3, wav = d / "a.mp3", d / "a.wav"
        _speak(normalize_for_tts(text), voice, mp3, rate)
        _to_wav(mp3, wav)
        return Response(content=wav.read_bytes(), media_type="audio/wav")
    except Exception as e:
        raise HTTPException(status_code=502, detail=str(e)[:200])
    finally:
        shutil.rmtree(d, ignore_errors=True)


class SelfTestRequest(BaseModel):
    gemini_key: str = ""
    gemini_model: str = "gemini-3.8-flash"


@app.post("/selftest")
def selftest(req: SelfTestRequest, x_app_token: Optional[str] = Header(default=None)):
    """Checks what the recap needs, one by one, and says in Burmese what to do
    when something fails. Every check is [{id, ok, note}]; nothing is stored."""
    _auth(x_app_token)
    checks = []

    def add(cid: str, ok: bool, note: str):
        checks.append({"id": cid, "ok": ok, "note": note})

    ff = _ffmpeg()
    try:
        p = subprocess.run([ff, "-version"], capture_output=True, text=True, timeout=20)
        add("ffmpeg", p.returncode == 0, "ရတယ်" if p.returncode == 0 else "ffmpeg မပွင့်ဘူး (requirements ထဲ imageio-ffmpeg ပါလား)")
    except Exception as e:
        add("ffmpeg", False, "ffmpeg မရှိဘူး: " + str(e)[:80])

    d = OUT_DIR / f"selftest_{secrets.token_hex(3)}"
    try:
        d.mkdir(parents=True, exist_ok=True)
        _speak("မင်္ဂလာပါ", DEFAULT_VOICE, d / "t.mp3")
        _to_wav(d / "t.mp3", d / "t.wav")
        add("edge_tts", True, f"မြန်မာအသံ လုပ်လို့ရတယ် ({_wav_ms(d / 't.wav')} ms)")
    except Exception as e:
        add("edge_tts", False, "Edge TTS မရဘူး (backend က အင်တာနက်ထွက်လို့ရလား၊ ခဏနေ ပြန်စမ်း): " + str(e)[:100])
    finally:
        shutil.rmtree(d, ignore_errors=True)

    if not req.gemini_key.strip():
        add("gemini", False, "Gemini key မပါဘူး (Settings မှာ ထည့်ပါ)")
    else:
        try:
            client = genai.Client(api_key=req.gemini_key.strip())
            r = client.models.generate_content(model=req.gemini_model, contents="Reply with the single word OK.")
            add("gemini", True, f"{req.gemini_model} ဖြေတယ်: {(r.text or '').strip()[:20]}")
        except Exception as e:
            msg = mask_key(str(e), req.gemini_key)
            hint = gemini_hint(msg)
            add("gemini", False, hint + " · " + msg[:120])
    return {"checks": checks}


# ---- Creator Studio ---------------------------------------------------------
STUDIO_DIR = Path(os.environ.get("DF_STUDIO_DIR") or (Path(tempfile.gettempdir()) / "dramaflow_studio"))
STUDIO_OPS = ("download", "split", "translate", "subtitle", "zip", "youtube", "transcribe", "render", "karaoke", "nursery")
STUDIO_FLAGS: dict = {}  # job id -> {"cancel": bool, "pause": bool}


class StudioUpload(BaseModel):
    name: str
    data_b64: str


class StudioRequest(BaseModel):
    op: str
    # download
    url: str = ""
    quality: str = "720"
    speed: str = "safe4"
    playlist: bool = False
    playlist_items: str = ""
    sub_langs: str = "auto"
    subtitles: bool = False
    auto_subs: bool = False
    # split / subtitle / translate
    file: str = ""
    split_mode: str = "count"
    count: int = 2
    ranges: str = ""
    srt_file: str = ""
    reencode: bool = False
    engine: str = "free"
    src_lang: str = "auto"
    gemini_key: str = ""
    gemini_model: str = "gemini-3.8-flash"
    sub_mode: str = "soft"
    font: str = "Padauk"
    font_size: int = 22
    # transcribe (Whisper)
    whisper_model: str = "small"
    whisper_vad: bool = True
    # render: the app's render job (see render_core.py)
    job: dict = {}
    # children's song video (karaoke + nursery ops)
    lyrics: str = ""
    karaoke_mode: str = "lyrics"
    ratio: str = "16:9"
    durations: List[float] = []
    # zip / youtube
    files: List[str] = []
    zip_name: str = ""
    confirm_rights: bool = False
    privacy: str = "private"
    title_template: str = "{name}"
    description: str = ""
    tags: str = ""
    category: str = "entertainment"
    kids: bool = False
    thumbnail: str = ""
    upload_cc: bool = False


class YoutubePaste(BaseModel):
    pasted: str


def _studio_gemini(req: StudioRequest):
    """Lines in, Burmese lines out, with the user's own Gemini key (used for this job only)."""
    client = genai.Client(api_key=req.gemini_key)

    def many(texts: List[str], lang: str) -> List[str]:
        prompt = sc.numbered_prompt(texts, lang)
        for attempt in range(3):
            try:
                r = client.models.generate_content(model=req.gemini_model, contents=prompt, config=types.GenerateContentConfig(temperature=0.3))
                return sc.parse_numbered_reply(r.text or "", len(texts))
            except Exception as e:
                msg = str(e)
                if any(k in msg for k in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and attempt < 2:
                    time.sleep(20 * (attempt + 1))
                    continue
                raise RuntimeError(mask_key(msg, req.gemini_key))
        return []

    return many


def run_studio_job(job_id: str, req: StudioRequest):
    flags = STUDIO_FLAGS.setdefault(job_id, {"cancel": False, "pause": False})

    def detail(**kw):
        with LOCK:
            JOBS[job_id].setdefault("detail", {}).update(kw)

    env = so.Env(
        STUDIO_DIR, _ffmpeg(),
        lambda stage, pct: _set(job_id, stage=stage, progress=min(100, max(0, int(pct)))),
        lambda: flags["cancel"], lambda: flags["pause"], detail,
        _studio_gemini(req) if req.engine == "gemini" and req.gemini_key.strip() else None,
    )
    try:
        STUDIO_DIR.mkdir(parents=True, exist_ok=True)
        _set(job_id, status="running", stage="စတင်နေတယ်", progress=1)
        fn = {"download": so.op_download, "split": so.op_split, "translate": so.op_translate,
              "subtitle": so.op_subtitle_video, "zip": so.op_zip, "youtube": so.op_youtube, "transcribe": so.op_transcribe, "render": so.op_render, "karaoke": so.op_karaoke, "nursery": so.op_nursery}[req.op]
        result = fn(env, req)
        _set(job_id, status="done", stage="ပြီးပြီ", progress=100, result=result)
    except so.Canceled:
        _set(job_id, status="canceled", stage="ရပ်လိုက်ပြီ")
    except Exception as e:  # shown in the app, with the key hidden
        _set(job_id, status="error", stage="မအောင်မြင်ဘူး", error=mask_key(str(e), req.gemini_key)[:500])
    finally:
        flags["pause"] = False
        detail(paused=False)


@app.get("/studio/files")
def studio_files(x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    STUDIO_DIR.mkdir(parents=True, exist_ok=True)
    try:
        free = round(shutil.disk_usage(STUDIO_DIR).free / 1024**3, 1)
    except OSError:
        free = None
    return {"files": so.list_library(STUDIO_DIR), "free_gb": free, "aria2c": bool(shutil.which("aria2c")), "youtube": so.yt_status()}


@app.get("/studio/files/{name}")
def studio_file(name: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    try:
        p = so.file_path(STUDIO_DIR, name)
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))
    return FileResponse(str(p), filename=p.name, media_type="application/octet-stream")


@app.delete("/studio/files/{name}")
def studio_delete(name: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    try:
        so.delete_file(STUDIO_DIR, name)
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))
    return {"ok": True}


@app.post("/studio/upload")
def studio_upload(req: StudioUpload, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    try:
        return {"name": so.save_upload(STUDIO_DIR, req.name, req.data_b64)}
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))


@app.post("/studio/jobs")
def studio_create(req: StudioRequest, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if req.op not in STUDIO_OPS:
        raise HTTPException(status_code=400, detail="op မမှန်ဘူး")
    if req.op == "youtube" and not req.confirm_rights:
        raise HTTPException(status_code=400, detail="YouTube ပေါ် တင်မယ့် ဗီဒီယိုက ကိုယ်ပိုင် / တင်ခွင့်ရှိတာ ဖြစ်ကြောင်း အတည်ပြုပါ")
    if req.op == "youtube" and req.privacy not in sc.YT_PRIVACY:
        raise HTTPException(status_code=400, detail="privacy မမှန်ဘူး")
    if req.op == "translate" and req.engine == "gemini" and not req.gemini_key.strip():
        raise HTTPException(status_code=400, detail="Gemini key မပါဘူး (Settings မှာ ထည့်ပါ)")
    clean_old()
    with LOCK:
        if any(j["status"] in ("queued", "running") for j in JOBS.values()):
            raise HTTPException(status_code=409, detail="တခြား job တစ်ခု လုပ်နေတုန်းပါ")
        job_id = f"studio_{int(time.time())}_{secrets.token_hex(2)}"
        JOBS[job_id] = {"status": "queued", "stage": "စောင့်နေတယ်", "progress": 0, "created": time.time(), "error": "",
                        "op": req.op, "detail": {}, "result": {"files": [], "links": [], "notes": [], "errors": []}}
        STUDIO_FLAGS[job_id] = {"cancel": False, "pause": False}
    threading.Thread(target=run_studio_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id}


def _studio_job(job_id: str):
    j = JOBS.get(job_id) if job_id.startswith("studio_") else None
    if j is None:
        raise HTTPException(status_code=404, detail="Job မတွေ့ပါ")
    return j


@app.get("/studio/jobs/{job_id}")
def studio_status(job_id: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    with LOCK:
        j = _studio_job(job_id)
        return {k: (dict(v) if isinstance(v, dict) else v) for k, v in j.items() if k != "created"}


@app.post("/studio/jobs/{job_id}/{action}")
def studio_control(job_id: str, action: str, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    if action not in ("cancel", "pause", "resume"):
        raise HTTPException(status_code=404, detail="Not found")
    with LOCK:
        j = _studio_job(job_id)
        if j["status"] not in ("queued", "running"):
            raise HTTPException(status_code=409, detail="Job ပြီးသွားပြီ")
    flags = STUDIO_FLAGS.setdefault(job_id, {"cancel": False, "pause": False})
    if action == "cancel":
        flags["cancel"] = True
        flags["pause"] = False
    else:
        flags["pause"] = action == "pause"
    return {"ok": True}


@app.get("/studio/youtube/status")
def studio_yt_status(x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    return so.yt_status()


@app.post("/studio/youtube/auth-link")
def studio_yt_link(x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    try:
        return {"url": so.yt_auth_link()}
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except RuntimeError as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/studio/youtube/auth-complete")
def studio_yt_complete(req: YoutubePaste, x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    try:
        return so.yt_auth_complete(req.pasted)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))


@app.post("/studio/youtube/unlink")
def studio_yt_unlink(x_app_token: Optional[str] = Header(default=None)):
    _auth(x_app_token)
    so.yt_unlink()
    return so.yt_status()

In [ ]:
%%writefile /content/dramaflow_backend/recap_core.py
"""Recap job logic with no web framework and no network, so it can be tested.

The app sends GROUPS. Each group already holds the prompt the app built (the
same prompt the manual flow copies to Gemini), an optional contact-sheet JPEG
and, for a whole video, the time range to look at. This module asks Gemini for
each group, reads the "=== PART n ===" parts out of the reply, and makes the
voice for each part. Gemini and the voice are passed in as functions.
"""
import re
import time
from typing import Callable, Dict, List, Optional, Tuple

PART_HEAD = re.compile(r"^[ \t]*[=\-#*\s]*PART\s*(\d+)[=\-#*\s:：]*$", re.IGNORECASE | re.MULTILINE)


def parse_reply(reply: str) -> Dict[int, str]:
    """Part number -> text. Text before the first part line is ignored."""
    text = (reply or "").replace("\r\n", "\n").replace("```", "")
    heads = list(PART_HEAD.finditer(text))
    out: Dict[int, str] = {}
    for i, h in enumerate(heads):
        n = int(h.group(1))
        end = heads[i + 1].start() if i + 1 < len(heads) else len(text)
        body = text[h.end():end].strip()
        if n >= 1 and body:
            out[n] = body
    return out


def gemini_hint(message: str) -> str:
    """What to do next, in Burmese, for a Gemini error text (used by the self-test)."""
    m, low = message, message.lower()
    if "location is not supported" in low or "user location" in low:
        return "Backend (Colab) ရှိတဲ့ နေရာကို Google က မခွင့်ပြုဘူး — Colab ရဲ့ Runtime > Disconnect and delete runtime လုပ်ပြီး အစကနေ ပြန် run ပါ"
    if "404" in m or "NOT_FOUND" in m:
        return "model နာမည် မရှိတော့ဘူး — model ကွက်မှာ gemini-3.8-flash လို နာမည်ပြောင်းပါ"
    if "429" in m or "RESOURCE_EXHAUSTED" in m:
        return "quota ပြည့်နေတယ် — ခဏစောင့်ပါ၊ aistudio.google.com/rate-limit မှာ စစ်ပါ"
    if "503" in m or "UNAVAILABLE" in m:
        return "Google ဘက် model လူသုံးများနေတယ် (ခဏဖြစ်တာ) — ခဏနေ ပြန်စမ်းပါ၊ အရန် model ထည့်ထားရင် Auto က အလိုအလျောက် ပြောင်းစမ်းမယ်"
    if "api key" in low or "400" in m or "403" in m or "PERMISSION" in m:
        return "key မှန်လား၊ Gemini API ဖွင့်ထားလား စစ်ပါ"
    return "မသိတဲ့ အမှား"


def mask_key(message: str, key: str) -> str:
    """Never send the user's API key back in an error text."""
    if key and len(key) >= 8:
        return message.replace(key, "***")
    return message


def part_targets(groups: List[dict]) -> Dict[int, int]:
    """Part number -> how long its video part is (ms), from each group's
    part_ms list (one entry per part, first..last). Missing or wrong-length
    lists give no target for that group."""
    out: Dict[int, int] = {}
    for g in groups:
        ms = g.get("part_ms") or []
        first, last = int(g["first"]), int(g["last"])
        if len(ms) != last - first + 1:
            continue
        for i, v in enumerate(ms):
            try:
                v = int(v)
            except (TypeError, ValueError):
                continue
            if v > 0:
                out[first + i] = v
    return out


GeminiFn = Callable[[dict], str]
TtsFn = Callable[[int, str, int], Tuple[int, str]]  # (part number, text, target ms or 0) -> (milliseconds, file name)
ProgressFn = Callable[[str, int], None]


MAX_SPEEDUP = 1.15  # the voice is sped up at most this much to fit its video part
LONG_WARN = 1.10  # still this much longer than the part after speeding up -> warn
SHORT_WARN = 0.5  # less than this share of the part -> warn
CONTEXT_CHARS = 300


def fit_tempo(audio_ms: int, target_ms: int, max_speed: float = MAX_SPEEDUP) -> float:
    """Speed factor (1.0 = unchanged) that makes a voice that is too long fit
    its video part. Never slows the voice down and never goes above max_speed."""
    if target_ms <= 0 or audio_ms <= 0 or audio_ms <= target_ms * 1.02:
        return 1.0
    return round(min(audio_ms / target_ms, max_speed), 3)


def fit_warning(n: int, audio_ms: int, target_ms: int) -> str:
    """A short Burmese note when the voice of part n does not fit its video
    part, or an empty string."""
    if target_ms <= 0 or audio_ms <= 0:
        return ""
    a, t = audio_ms / 1000, target_ms / 1000
    if audio_ms > target_ms * LONG_WARN:
        return f"အပိုင်း {n}: အသံ {a:.1f} စက္ကန့်၊ ဗီဒီယိုအပိုင်း {t:.1f} စက္ကန့် — အသံက ရှည်နေတယ် (script ကို တိုစေပါ)"
    if audio_ms < target_ms * SHORT_WARN:
        return f"အပိုင်း {n}: အသံ {a:.1f} စက္ကန့်၊ ဗီဒီယိုအပိုင်း {t:.1f} စက္ကန့် — အသံက တိုနေတယ် (အပိုင်းထဲ တိတ်နေမယ်)"
    return ""


def _tail(text: str, limit: int = CONTEXT_CHARS) -> str:
    t = " ".join(text.split())
    return t if len(t) <= limit else t[-limit:]


def with_context(prompt: str, previous_text: str) -> str:
    """Adds how the story stood at the end of the previous group, so names and
    events stay consistent from one request to the next."""
    if not previous_text.strip():
        return prompt
    return (
        prompt.rstrip()
        + "\n\nCONTEXT (the previous part of this recap ended like this. Do not repeat it; "
        + "only keep names and events consistent with it):\n"
        + _tail(previous_text)
        + "\n"
    )


def run_recap(
    groups: List[dict],
    gemini: GeminiFn,
    tts: Optional[TtsFn],
    progress: ProgressFn,
    gap_seconds: float = 6.0,
    sleep: Callable[[float], None] = time.sleep,
) -> dict:
    """Returns {"parts": [{"n", "text", "audio_ms", "audio_file"}], "warnings": [...]}.
    A group that fails is reported as a warning and the rest go on; when nothing
    at all came out, ValueError is raised."""
    warnings: List[str] = []
    texts: Dict[int, str] = {}
    total = max(1, len(groups))
    for gi, g in enumerate(groups):
        first, last = int(g["first"]), int(g["last"])
        progress(f"Gemini က အပိုင်း {first}–{last} ရေးနေတယ်", 5 + int(55 * gi / total))
        if texts:
            g = dict(g, prompt=with_context(g["prompt"], texts[max(texts)]))
        try:
            reply = gemini(g)
        except Exception as e:  # reported, the next group still runs
            warnings.append(f"အပိုင်း {first}–{last} မရဘူး: {str(e)[:160]}")
            continue
        got = parse_reply(reply)
        missing = []
        for n in range(first, last + 1):
            if n in got:
                texts[n] = got[n]
            else:
                missing.append(n)
        if missing:
            warnings.append("Gemini က မရေးပေးတဲ့ အပိုင်း: " + ", ".join(str(n) for n in missing))
        if gi + 1 < len(groups):
            sleep(gap_seconds)
    if not texts:
        raise ValueError("Gemini က script တစ်ပိုင်းမှ မပေးဘူး။ " + (warnings[0] if warnings else ""))
    targets = part_targets(groups)
    parts = []
    order = sorted(texts)
    for idx, n in enumerate(order):
        audio_ms, audio_file = 0, ""
        target = targets.get(n, 0)
        if tts is not None:
            progress(f"အသံ လုပ်နေတယ် {idx + 1}/{len(order)}", 62 + int(36 * idx / len(order)))
            try:
                audio_ms, audio_file = tts(n, texts[n], target)
                w = fit_warning(n, audio_ms, target)
                if w:
                    warnings.append(w)
            except Exception as e:
                warnings.append(f"အပိုင်း {n} အသံ မရဘူး: {str(e)[:120]}")
        parts.append({"n": n, "text": texts[n], "audio_ms": audio_ms, "target_ms": target, "audio_file": audio_file})
    return {"parts": parts, "warnings": warnings}

In [ ]:
%%writefile /content/dramaflow_backend/tts_text.py
"""Cleans narration text before it goes to the voice (Edge TTS).

Pure Python, no network. The goal is only to remove what a voice reads badly:
emoji, markdown marks, brackets with stage directions, and small plain numbers
(0-99), which are spelled as Burmese words. Everything else is left alone, so
the text the user reviewed is the text that is spoken.
"""
import re
import unicodedata

_UNITS = {1: "တစ်", 2: "နှစ်", 3: "သုံး", 4: "လေး", 5: "ငါး", 6: "ခြောက်", 7: "ခုနစ်", 8: "ရှစ်", 9: "ကိုး"}
_TENS = {1: "ဆယ်", 2: "နှစ်ဆယ်", 3: "သုံးဆယ်", 4: "လေးဆယ်", 5: "ငါးဆယ်", 6: "ခြောက်ဆယ်", 7: "ခုနစ်ဆယ်", 8: "ရှစ်ဆယ်", 9: "ကိုးဆယ်"}
_ZERO = "သုံည"


def number_words(n: int) -> str:
    """0..99 in Burmese words: 11 -> ဆယ့်တစ်, 25 -> နှစ်ဆယ့်ငါး."""
    if not 0 <= n <= 99:
        raise ValueError("only 0..99")
    if n == 0:
        return _ZERO
    if n < 10:
        return _UNITS[n]
    t, u = divmod(n, 10)
    if u == 0:
        return _TENS[t]
    # "ဆယ်" becomes "ဆယ့်" (asat after the dot below) when a unit follows
    return _TENS[t][:-1] + "\u1037\u103a" + _UNITS[u]


# A plain 1-2 digit whole number. Not part of a longer number, a decimal, a
# time (12:30), a date or range (3/4, 1-2) or a version (1.2.3).
_NUM = re.compile(r"(?<![\d.,:/\-])\d{1,2}(?![\d.,:/]|-\d)")
_BRACKETS = re.compile(r"[\(\[（【][^\)\]）】\n]{0,80}[\)\]）】]")
_MARKS = re.compile(r"[*_`#>~|=]+")
_SPACES = re.compile(r"[ \t ]+")


def _drop_symbols(text: str) -> str:
    out = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat in ("So", "Sk", "Cs", "Co", "Cn"):  # emoji, pictographs, private use
            continue
        if ch in "‍️​‌⁠﻿":  # joiners, variation selectors, zero width
            continue
        out.append(ch)
    return "".join(out)


def normalize_for_tts(text: str) -> str:
    """Returns the text the voice should read. Never returns an empty string
    for non-empty input: if everything would be removed, the original is kept."""
    original = (text or "").strip()
    t = original.replace("\r\n", "\n")
    t = _BRACKETS.sub(" ", t)  # (laughs) [music] and similar
    t = _drop_symbols(t)
    t = _MARKS.sub(" ", t)
    t = _NUM.sub(lambda m: number_words(int(m.group(0))), t)
    lines = [_SPACES.sub(" ", ln).strip() for ln in t.split("\n")]
    t = "\n".join(ln for ln in lines if ln)
    return t if t else original

In [ ]:
%%writefile /content/dramaflow_backend/ingest_core.py
"""Link -> video, the logic with no network and no web framework.

The backend (main.py) downloads the video of a link the user gave (YouTube,
TikTok, Red Note ...) with yt-dlp, puts it into the user's OWN Gemini account
(Files API) and answers with the address Gemini needs. The download is a
private working copy on the backend; it is deleted after the upload. Nothing
here touches the network, so it is tested on its own (test_ingest_core.py).
"""
import ipaddress
import os
import re
import socket
from pathlib import Path
from typing import Callable, Optional, Tuple
from urllib.parse import urlparse

MAX_SECONDS = 3 * 3600  # longer than this is refused (disk and Gemini limits)
MAX_BYTES = 2 * 1024**3  # Gemini Files API accepts up to 2 GB per file
GEMINI_MIMES = {
    ".mp4": "video/mp4",
    ".mov": "video/mov",
    ".webm": "video/webm",
    ".avi": "video/avi",
    ".mpeg": "video/mpeg",
    ".mpg": "video/mpg",
    ".flv": "video/x-flv",
    ".wmv": "video/wmv",
    ".3gp": "video/3gpp",
}

# Subtitles that come with a video (only the ones the uploader made, not the
# machine captions). The first language in the list that exists wins.
SUB_LANGS = ["zh-Hans", "zh-CN", "zh", "zh-Hant", "zh-TW", "en", "my"]
MAX_SRT_CHARS = 300_000
SUB_SUFFIXES = (".srt", ".vtt", ".ass", ".ssa", ".ttml", ".json", ".json3", ".srv1", ".srv2", ".srv3", ".xml", ".lrc", ".txt")
MAX_COOKIE_BYTES = 400 * 1024
_COOKIE_DOMAINS = {
    "youtube": ("youtube.com", "google.com"),
    "tiktok": ("tiktok.com",),
    "rednote": ("xiaohongshu.com", "xhscdn.com"),
    "douyin": ("douyin.com",),
    "bilibili": ("bilibili.com",),
}

_URL = re.compile(r"https?://[^\s<>\"'“”‘’，。）)】」]+", re.I)


def extract_url(text: str) -> str:
    """The first web address in [text] (share texts from TikTok / Red Note
    carry words around the link). Empty when there is none."""
    m = _URL.search(text or "")
    return m.group(0).rstrip(".,;:!?]}'\"") if m else ""


def classify(url: str) -> str:
    host = (urlparse(url).hostname or "").lower()

    def on(*names):
        return any(host == n or host.endswith("." + n) for n in names)

    if on("youtube.com", "youtu.be"):
        return "youtube"
    if on("tiktok.com"):
        return "tiktok"
    if on("xiaohongshu.com", "xhslink.com"):
        return "rednote"
    if on("douyin.com", "iesdouyin.com"):
        return "douyin"
    if on("bilibili.com", "b23.tv"):
        return "bilibili"
    return "other"


def cookies_problem(text: str) -> str:
    """"" when [text] looks like a Netscape cookies.txt (what yt-dlp reads),
    otherwise what is wrong, in Burmese."""
    t = (text or "").lstrip("\ufeff").strip()
    if not t:
        return "cookies.txt ဖိုင်က အလွတ်ဖြစ်နေတယ်"
    if len(t.encode("utf-8", "ignore")) > MAX_COOKIE_BYTES:
        return "cookies.txt ဖိုင် ကြီးလွန်းတယ် (400 KB ထက် မကျော်ရ)"
    if t.startswith(("{", "[")):
        return "cookies.txt က JSON ပုံစံဖြစ်နေတယ် — Extension မှာ Netscape format ရွေးပြီး ပြန်ထုတ်ပါ"
    rows = 0
    for ln in t.splitlines():
        ln = ln.strip()
        if not ln or (ln.startswith("#") and not ln.startswith("#HttpOnly_")):
            continue
        if len(ln.split("\t")) >= 7:
            rows += 1
    if rows == 0:
        return "cookies.txt ထဲမှာ cookie တစ်ကြောင်းမှ မတွေ့ဘူး (Netscape format ဖြစ်ရမယ်၊ tab နဲ့ ခြားထားတာ)"
    return ""


def cookie_note(text: str, kind: str) -> str:
    """A warning when the cookies are not for the website of the link ("" if fine or unknown)."""
    want = _COOKIE_DOMAINS.get(kind)
    if not want:
        return ""
    low = (text or "").lower()
    if any(d in low for d in want):
        return ""
    return "cookies.txt ထဲမှာ ဒီ website (%s) ရဲ့ cookie မတွေ့ဘူး" % want[0]


def pick_subtitle(names) -> Tuple[Optional[str], str]:
    """From the files a download left (v.zh-Hans.srt, v.en.srt ...) the best
    subtitle: Chinese first, then English, Burmese, then anything. Danmaku /
    live chat are not subtitles. Returns (file name, language) or (None, "")."""
    best = None
    for n in names:
        m = re.fullmatch(r"v\.(.+)\.srt", n or "", re.I)
        if not m:
            continue
        lang = m.group(1)
        if re.search(r"danmaku|live_chat", lang, re.I):
            continue
        if lang in SUB_LANGS:
            rank = SUB_LANGS.index(lang)
        elif lang.lower().startswith("zh"):
            rank = len(SUB_LANGS)
        else:
            rank = len(SUB_LANGS) + 1
        key = (rank, lang)
        if best is None or key < best[0]:
            best = (key, n, lang)
    return (best[1], best[2]) if best else (None, "")


def clean_srt(text: str) -> str:
    """The subtitle text when it looks like an SRT and is not huge, else ""."""
    t = (text or "").lstrip("\ufeff").strip()
    if "-->" not in t or len(t) > MAX_SRT_CHARS:
        return ""
    return t


def cookies_source() -> Optional[Path]:
    """An optional cookies.txt the user put on the backend machine themselves (Colab: /content/cookies.txt,
    or the path in DF_COOKIES). It never goes through the app."""
    for c in (os.environ.get("DF_COOKIES"), "/content/cookies.txt", str(Path(__file__).resolve().parent / "cookies.txt")):
        if c and Path(c).is_file():
            return Path(c)
    return None


def check_url(url: str, resolve: Callable = socket.getaddrinfo) -> Tuple[bool, str]:
    """Only public http(s) addresses. The backend runs on a shared machine, so
    addresses that point inside it (localhost, private networks) are refused."""
    try:
        u = urlparse((url or "").strip())
    except ValueError:
        return False, "link ပုံစံ မမှန်ဘူး"
    if u.scheme not in ("http", "https") or not u.hostname:
        return False, "http:// သို့မဟုတ် https:// နဲ့စတဲ့ link ထည့်ပါ"
    host = u.hostname
    try:
        ips = [ipaddress.ip_address(host)]
    except ValueError:
        try:
            ips = [ipaddress.ip_address(info[4][0]) for info in resolve(host, None)]
        except Exception:
            return False, "link ရဲ့ website ကို ရှာမတွေ့ဘူး (စာလုံးပေါင်း စစ်ပါ)"
    for ip in ips:
        if ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_reserved or ip.is_multicast or ip.is_unspecified:
            return False, "ဒီ link က အများသုံး website မဟုတ်ဘူး"
    return True, ""


def mime_for(path: str) -> Optional[str]:
    m = re.search(r"(\.[A-Za-z0-9]+)$", path or "")
    return GEMINI_MIMES.get(m.group(1).lower()) if m else None


def parse_ffmpeg_duration(stderr: str) -> int:
    """Milliseconds from the 'Duration: 00:12:34.56' line of `ffmpeg -i`. 0 if none."""
    m = re.search(r"Duration:\s*(\d+):(\d{2}):(\d{2})(?:\.(\d+))?", stderr or "")
    if not m:
        return 0
    h, mi, s = int(m.group(1)), int(m.group(2)), int(m.group(3))
    frac = int((m.group(4) or "0").ljust(3, "0")[:3])
    return ((h * 60 + mi) * 60 + s) * 1000 + frac


def ingest_hint(message: str, kind: str = "other", has_cookies: bool = False) -> str:
    """What to do next, in Burmese, for an error from the download or the upload."""
    low = (message or "").lower()
    if "sign in to confirm" in low or "not a bot" in low:
        if has_cookies:
            return ("cookies.txt ထည့်ထားပေမဲ့ YouTube က ဆက်တားနေတယ် — cookie သက်တမ်းကုန်နေနိုင်တယ်။ Chrome မှာ YouTube ကို login ဝင်ထားပြီး "
                    "cookies.txt ကို အသစ်ပြန်ထုတ်ပြီး Colab ထဲ ပြန်တင်ပါ (Colab runtime ကိုလည်း ပြန်စကြည့်ပါ)")
        return ("YouTube က Colab ကို 'robot မဟုတ်ကြောင်း အတည်ပြုပါ' လို့ တားထားတယ် — ဖြေရှင်းနည်း: Chrome မှာ YouTube ကို login ဝင်ပြီး "
                "cookies.txt (Netscape format) ထုတ်၊ Colab ရဲ့ ဘယ်ဘက် Files ထဲ /content/cookies.txt အဖြစ် တင်ပြီး ပြန်စမ်းပါ။ "
                "(ဒါမှမဟုတ် ခဏနေစမ်း / runtime ပြန်စ / ဗီဒီယိုဖိုင်ကို ပုံမှန်အတိုင်း ရွေးပါ)")
    if "private" in low or "login" in low or "log in" in low or "members-only" in low or "members only" in low:
        return "ဒီဗီဒီယိုက private / login လိုတယ် — အများမြင်လို့ရတဲ့ link ကို ထည့်ပါ"
    if "unsupported url" in low:
        return "ဒီ link ပုံစံကို မဖတ်တတ်ဘူး — ဗီဒီယိုစာမျက်နှာ တိုက်ရိုက် link ကို share ကနေ ပြန်ကူးပါ (channel/profile link မဟုတ်ရ)"
    if "video unavailable" in low or "has been removed" in low or "not available" in low or "404" in low:
        return "ဗီဒီယို မရှိတော့ဘူး / ဒီနိုင်ငံမှာ မရဘူး (link မှန်လား စစ်ပါ)"
    if "429" in low or "too many requests" in low or "403" in low or "forbidden" in low:
        hint = "website က Colab ကို ခဏ တားထားတယ် — ခဏနေ ပြန်စမ်းပါ"
        if kind in ("tiktok", "rednote", "douyin"):
            hint += " (ဒီ website တွေက တားလေ့ရှိတယ်၊ မရရင် ဗီဒီယိုဖိုင်ကို ရွေးတင်ပါ)"
        return hint
    if "ဖိုင် ကြီး" in (message or "") or "larger than" in low or "max-filesize" in low:
        return "ဗီဒီယိုက ကြီးလွန်းတယ် (၂ GB ကျော်)"
    if "user location" in low or "location is not supported" in low:
        return "Colab ရှိတဲ့ နေရာကို Google က မခွင့်ပြုဘူး — Colab runtime ကို ဖျက်ပြီး ပြန်စပါ"
    if "api key" in low or "permission" in low or "401" in low:
        return "Gemini key မှန်လား၊ Gemini API ဖွင့်ထားလား စစ်ပါ"
    return ""


def run_ingest(
    url: str,
    download: Callable[[str, Callable[[int], None]], dict],
    upload: Callable[[str, str], Tuple[str, str]],
    progress: Callable[[str, int], None],
) -> dict:
    """download(url, on_percent) -> {"path", "title", "duration_ms"};
    upload(path, mime) -> (file_uri, mime). Returns what the app needs."""
    ok, why = check_url(url)
    if not ok:
        raise ValueError(why)
    kind = classify(url)
    progress("ဗီဒီယို ဆွဲနေတယ်", 5)
    got = download(url, lambda pct: progress("ဗီဒီယို ဆွဲနေတယ် %d%%" % max(0, min(100, pct)), 5 + int(max(0, min(100, pct)) * 0.55)))
    path = got["path"]
    mime = mime_for(path)
    if not mime:
        raise ValueError("ဒီဗီဒီယိုပုံစံ (%s) ကို Gemini က မလက်ခံဘူး" % (path.rsplit(".", 1)[-1] if "." in path else "?"))
    progress("Gemini ဆီ တင်နေတယ်", 62)
    uri, mime = upload(path, mime)
    progress("ပြီးပြီ", 100)
    out = {
        "file_uri": uri,
        "mime": mime,
        "duration_ms": int(got.get("duration_ms") or 0),
        "title": str(got.get("title") or "")[:120],
        "source": kind,
        "cookies": bool(got.get("cookies")),
    }
    srt = clean_srt(got.get("srt") or "")
    if srt:  # the uploader's own subtitles: the app can use them instead of paying for speech-to-text
        out["srt"] = srt
        out["srt_lang"] = str(got.get("srt_lang") or "")[:20]
    return out

In [ ]:
%%writefile /content/dramaflow_backend/studio_core.py
"""Creator Studio logic: no network, no web framework (tested in test_studio_core.py).

What the backend does for the "Studio" screen of the app, all on the backend
machine so a big video never has to travel through the tunnel to the phone:
download a link, cut a video into parts, translate subtitles, put the
subtitles into the MP4 (soft CC) and upload to the user's OWN YouTube channel.

Nothing here changes how a video looks (no zoom, flip, colour or pitch): parts
are cut as they are, subtitles are added as a separate track.
"""
import re
from typing import Callable, Dict, List, Optional, Tuple
from urllib.parse import parse_qs, urlparse

# ---- names and file kinds ---------------------------------------------------
_NAME_OK = re.compile(r"^[\w\-. ()က-႟]{1,120}$")
VIDEO_EXT = (".mp4", ".mov", ".webm", ".mkv", ".avi", ".flv", ".m4v")
AUDIO_EXT = (".mp3", ".m4a", ".wav", ".aac", ".ogg", ".opus")
SUB_EXT = (".srt",)


def safe_name(name: str) -> bool:
    """A plain file name (no folders, no '..'): the only kind the library serves or deletes."""
    n = name or ""
    return bool(_NAME_OK.match(n)) and not n.startswith(".") and ".." not in n and "/" not in n and "\\" not in n


def kind_of(name: str) -> str:
    low = (name or "").lower()
    if low.endswith(VIDEO_EXT):
        return "video"
    if low.endswith(AUDIO_EXT):
        return "audio"
    if low.endswith(SUB_EXT):
        return "srt"
    return "other"


def make_stem(title: str, fallback: str = "video") -> str:
    """A file name stem from a video title: letters (also Burmese / Chinese), digits, - and space."""
    t = re.sub(r"[^\w\-\sက-႟]", "_", title or "")
    t = re.sub(r"\s+", " ", t).strip(" ._")
    t = re.sub(r"_+", "_", t)
    return (t[:80].strip(" ._")) or fallback


def unique_name(stem: str, ext: str, existing) -> str:
    """stem.ext, or stem (2).ext, stem (3).ext ... when that name is taken."""
    have = set(existing)
    n = 1
    while True:
        cand = f"{stem}{'' if n == 1 else f' ({n})'}{ext}"
        if cand not in have:
            return cand
        n += 1


# ---- time and parts ---------------------------------------------------------
def parse_time(ts: str) -> Optional[float]:
    """'90', '1:30', '01:02:03', '1:30.5' -> seconds, None when it is not a time."""
    t = (ts or "").strip().replace(",", ".")
    if not t:
        return None
    parts = t.split(":")
    if len(parts) > 3:
        return None
    try:
        vals = [float(p) for p in parts]
    except ValueError:
        return None
    if any(v < 0 for v in vals):
        return None
    secs = 0.0
    for v in vals:
        secs = secs * 60 + v
    return secs


def fmt_time(sec: float) -> str:
    s = int(round(max(0.0, sec)))
    return f"{s // 3600:d}:{(s % 3600) // 60:02d}:{s % 60:02d}" if s >= 3600 else f"{s // 60:d}:{s % 60:02d}"


MIN_PART_SECONDS = 1.0
MAX_PARTS = 60


def equal_parts(duration: float, count: int) -> List[Tuple[float, float]]:
    """[count] parts of the same length that cover the whole video. A part is
    never shorter than a second, so too many parts are reduced."""
    if duration < MIN_PART_SECONDS:
        return []
    n = max(1, min(int(count), MAX_PARTS, int(duration // MIN_PART_SECONDS)))
    step = duration / n
    return [(round(i * step, 3), round(duration if i == n - 1 else (i + 1) * step, 3)) for i in range(n)]


def parse_ranges(text: str, duration: float) -> Tuple[List[Tuple[float, float]], str]:
    """'0:00-5:00, 5:00-12:30' (commas or new lines) -> ([(start, end)], error).
    An end past the video is cut to its end. The error is Burmese, '' when fine."""
    items = [x.strip() for x in re.split(r"[,\n;]+", text or "") if x.strip()]
    if not items:
        return [], "အပိုင်းအချိန် မထည့်ရသေးဘူး (ဥပမာ 0:00-5:00, 5:00-12:30)"
    out: List[Tuple[float, float]] = []
    for it in items:
        pieces = re.split(r"\s*(?:-|–|—|~|to)\s*", it, maxsplit=1)
        if len(pieces) != 2:
            return [], f"'{it}' ပုံစံ မမှန်ဘူး (စ-ဆုံး ဥပမာ 0:00-5:00)"
        a, b = parse_time(pieces[0]), parse_time(pieces[1])
        if a is None or b is None:
            return [], f"'{it}' ထဲက အချိန် မမှန်ဘူး"
        if duration > 0:
            b = min(b, duration)
        if a >= duration > 0:
            return [], f"'{it}' က ဗီဒီယိုအရှည် ({fmt_time(duration)}) ထက် ကျော်နေတယ်"
        if b - a < MIN_PART_SECONDS:
            return [], f"'{it}' က တိုလွန်းတယ် (အနည်းဆုံး ၁ စက္ကန့်၊ စချိန်<ဆုံးချိန်)"
        out.append((a, b))
    if len(out) > MAX_PARTS:
        return [], f"အပိုင်း {MAX_PARTS} ခုထက် မများရ"
    return out, ""


# ---- SRT --------------------------------------------------------------------
Cue = Tuple[int, int, str]  # start ms, end ms, text
_TIME = re.compile(r"(\d+):(\d{2}):(\d{2})[,.](\d{1,3})\s*-->\s*(\d+):(\d{2}):(\d{2})[,.](\d{1,3})")


def _ms(h, m, s, f) -> int:
    return ((int(h) * 60 + int(m)) * 60 + int(s)) * 1000 + int(str(f).ljust(3, "0")[:3])


def parse_srt(text: str) -> List[Cue]:
    cues: List[Cue] = []
    blocks = re.split(r"\n\s*\n", (text or "").lstrip("﻿").replace("\r\n", "\n").replace("\r", "\n").strip())
    for b in blocks:
        lines = [ln for ln in b.split("\n")]
        for i, ln in enumerate(lines):
            m = _TIME.search(ln)
            if m:
                body = "\n".join(x.strip() for x in lines[i + 1:] if x.strip())
                if body:
                    cues.append((_ms(*m.groups()[:4]), _ms(*m.groups()[4:]), body))
                break
    return cues


# ---- speech to text (Whisper) -----------------------------------------------
WHISPER_MODELS = ("tiny", "base", "small", "medium", "large-v3")
_WHISPER_LANG = re.compile(r"^[a-z]{2,3}$")


def whisper_lang(code: str) -> Optional[str]:
    """'auto' / empty / anything odd -> None (Whisper finds the language); 'zh-CN' -> 'zh'."""
    c = (code or "").strip().lower()
    if c in ("", "auto"):
        return None
    c = c.split("-")[0].split("_")[0]
    return c if _WHISPER_LANG.match(c) else None


def whisper_model_name(name: str) -> str:
    return name if name in WHISPER_MODELS else "small"


def segments_to_cues(segments, min_ms: int = 400) -> List[Cue]:
    """Whisper segments (objects with start / end in seconds and text) -> SRT cues: empty text dropped, every cue at least
    min_ms long, and no two cues overlapping (the earlier one is shortened)."""
    out: List[Cue] = []
    for seg in segments:
        text = " ".join(str(getattr(seg, "text", "") or "").split())
        if not text:
            continue
        a = max(0, int(round(float(getattr(seg, "start", 0.0)) * 1000)))
        b = int(round(float(getattr(seg, "end", 0.0)) * 1000))
        if b < a + min_ms:
            b = a + min_ms
        if out and a < out[-1][1]:
            ps, pe, pt = out[-1]
            if a - ps >= 200:
                out[-1] = (ps, a, pt)
            else:
                a = pe
                if b < a + min_ms:
                    b = a + min_ms
        out.append((a, b, text))
    return out


def _stamp(ms: int) -> str:
    ms = max(0, int(ms))
    return f"{ms // 3600000:02d}:{(ms // 60000) % 60:02d}:{(ms // 1000) % 60:02d},{ms % 1000:03d}"


def serialize_srt(cues: List[Cue]) -> str:
    return "\n".join(f"{i}\n{_stamp(a)} --> {_stamp(b)}\n{t}\n" for i, (a, b, t) in enumerate(cues, 1))


def slice_srt(cues: List[Cue], start_s: float, end_s: float) -> List[Cue]:
    """The cues that overlap [start_s, end_s], moved so the part starts at 0."""
    a, b = int(start_s * 1000), int(end_s * 1000)
    out: List[Cue] = []
    for s, e, t in cues:
        if e <= a or s >= b:
            continue
        out.append((max(s, a) - a, min(e, b) - a, t))
    return out


# ---- translation ------------------------------------------------------------
def batches(texts: List[str], max_chars: int = 2800, max_items: int = 40) -> List[List[int]]:
    """Indexes of [texts] in groups that are not too long for one request."""
    out: List[List[int]] = []
    cur: List[int] = []
    size = 0
    for i, t in enumerate(texts):
        add = len(t) + 8
        if cur and (size + add > max_chars or len(cur) >= max_items):
            out.append(cur)
            cur, size = [], 0
        cur.append(i)
        size += add
    if cur:
        out.append(cur)
    return out


def numbered_prompt(texts: List[str], src_hint: str = "") -> str:
    lines = "\n".join(f"{i}|{t.replace(chr(10), ' / ')}" for i, t in enumerate(texts, 1))
    src = f" ({src_hint})" if src_hint and src_hint != "auto" else ""
    return (
        f"Translate these subtitle lines{src} into natural spoken Burmese (Myanmar Unicode). "
        "Keep every line short, keep names consistent, do not explain. "
        "Answer with exactly the same numbering, one line per input line, in the form `N|translation`, and nothing else.\n\n" + lines
    )


def parse_numbered_reply(reply: str, count: int) -> List[str]:
    """The `N|text` lines of an answer as a list of [count] texts; a line that is
    missing stays empty."""
    got: Dict[int, str] = {}
    last = 0
    for ln in (reply or "").replace("\r", "").split("\n"):
        m = re.match(r"^\s*`?(\d+)\s*[|:)]\s*(.*?)`?\s*$", ln)
        if m and 1 <= int(m.group(1)) <= count:
            last = int(m.group(1))
            got[last] = m.group(2).strip()
        elif last and ln.strip() and not re.match(r"^\s*\d+\s*[|:)]", ln):
            got[last] = (got[last] + " " + ln.strip()).strip()
    return [got.get(i, "") for i in range(1, count + 1)]


def translate_cues(cues: List[Cue], translate_many: Callable[[List[str]], List[str]], progress: Optional[Callable[[int, int], None]] = None) -> Tuple[List[Cue], int]:
    """Translates every cue. [translate_many] takes a few lines and answers the
    same number of lines. A group that fails, or answers a wrong count, is done
    line by line; a line that still fails keeps its original text. Returns the
    new cues and how many lines stayed untranslated."""
    texts = [c[2] for c in cues]
    out = list(texts)
    failed = 0
    groups = batches(texts)
    for gi, idxs in enumerate(groups):
        part = [texts[i] for i in idxs]
        res: Optional[List[str]] = None
        try:
            r = translate_many(part)
            if isinstance(r, list) and len(r) == len(part):
                res = r
        except Exception:
            res = None
        if res is None:
            res = []
            for t in part:
                try:
                    one = translate_many([t])
                    res.append(one[0] if one else "")
                except Exception:
                    res.append("")
        for i, tr in zip(idxs, res):
            tr = (tr or "").strip()
            if tr:
                out[i] = tr
            else:
                failed += 1
        if progress:
            progress(gi + 1, len(groups))
    return [(c[0], c[1], out[i]) for i, c in enumerate(cues)], failed


# ---- ffmpeg commands --------------------------------------------------------
def split_cmd(ff: str, src: str, start: float, dur: float, out: str, reencode: bool) -> List[str]:
    """One part, cut as it is. Stream copy is fast but cuts at the nearest key
    frame; re-encoding cuts exactly."""
    cmd = [ff, "-y", "-ss", f"{start:.3f}", "-t", f"{dur:.3f}", "-i", src]
    if reencode:
        cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart"]
    else:
        cmd += ["-c", "copy", "-avoid_negative_ts", "make_zero", "-movflags", "+faststart"]
    return cmd + [out]


def mux_cc_cmd(ff: str, video: str, srt: str, out: str, lang: str = "mya") -> List[str]:
    """The subtitles as a separate (soft) track in the MP4; the picture and sound are copied untouched."""
    return [ff, "-y", "-i", video, "-i", srt, "-map", "0:v:0", "-map", "0:a?", "-map", "1:0", "-c:v", "copy", "-c:a", "copy",
            "-c:s", "mov_text", "-metadata:s:s:0", f"language={lang}", "-metadata:s:s:0", "title=Myanmar", "-disposition:s:0", "default",
            "-movflags", "+faststart", out]


# ---- YouTube ----------------------------------------------------------------
YT_CATEGORIES = {"film": "1", "entertainment": "24", "people": "22", "education": "27", "howto": "26", "news": "25", "music": "10", "gaming": "20"}
YT_PRIVACY = ("private", "unlisted", "public")


def format_title(template: str, name: str, idx: int, total: int) -> str:
    """{name}, {n}, {total} in the template; '<' and '>' are not allowed by YouTube; 100 characters at most."""
    t = (template or "{name}").replace("{name}", name).replace("{n}", str(idx)).replace("{total}", str(total))
    t = re.sub(r"[<>]", "", t).strip()
    t = t[:100].strip()
    return t or name[:100] or "video"


def clean_tags(text: str) -> List[str]:
    """Comma separated tags without '<' '>', all together under ~450 characters."""
    out, size = [], 0
    for raw in re.split(r"[,\n]+", text or ""):
        t = re.sub(r"[<>]", "", raw).strip()
        if not t or size + len(t) + 1 > 450:
            continue
        out.append(t)
        size += len(t) + 1
    return out


def clean_description(text: str) -> str:
    return re.sub(r"[<>]", "", text or "")[:4900]


def extract_oauth_code(pasted: str) -> str:
    """The `code` from the address the browser ended on after Google's login
    (http://localhost/?code=...&scope=...), or the pasted text itself when it is
    only the code."""
    t = (pasted or "").strip()
    if not t:
        return ""
    if "code=" in t:
        try:
            q = parse_qs(urlparse(t).query)
            if q.get("code"):
                return q["code"][0].strip()
        except ValueError:
            pass
        m = re.search(r"[?&]code=([^&\s#]+)", t)
        if m:
            return m.group(1)
    return t if re.fullmatch(r"[\w\-./]{10,}", t) else ""


def youtube_error_hint(message: str) -> str:
    low = (message or "").lower()
    if "quotaexceeded" in low or "quota" in low:
        return "YouTube API quota ပြည့်သွားပြီ (တစ်ရက် ဗီဒီယို ၆ ပုဒ်လောက်ပဲ ရတတ်တယ်) — မနက်ဖြန် ပြန်စမ်းပါ"
    if "uploadlimitexceeded" in low:
        return "ဒီ channel ရဲ့ တစ်ရက် upload အကန့်အသတ် ပြည့်သွားပြီ"
    if "invalid_grant" in low or "expired" in low or "revoked" in low:
        return "YouTube login သက်တမ်းကုန်သွားပြီ — Studio > YouTube မှာ ပြန် login ဝင်ပါ"
    if "forbidden" in low or "403" in low or "insufficient" in low:
        return "YouTube က ခွင့်မပြုဘူး (scope/ခွင့်ပြုချက် မလောက်ဘူး၊ ပြန် login ဝင်ပါ။ OAuth app က Testing ဖြစ်ရင် Test users ထဲ သင့် email ထည့်ထားရမယ်)"
    if "unauthorized" in low or "401" in low:
        return "YouTube login မမှန်တော့ဘူး — ပြန် login ဝင်ပါ"
    return ""


# ---- downloading ------------------------------------------------------------
# key -> (yt-dlp format, what it makes). "mp3" and "srt" are not video downloads.
QUALITIES = {
    "best": ("bestvideo+bestaudio/best", "video"),
    "1080": ("bestvideo[height<=1080]+bestaudio/best[height<=1080]/bestvideo+bestaudio/best", "video"),
    "1080hevc": ("bestvideo[height<=1080][vcodec^=hev]+bestaudio/bestvideo[height<=1080][vcodec^=hvc]+bestaudio/bestvideo[height<=1080]+bestaudio/best", "video"),
    "720": ("bestvideo[height<=720]+bestaudio/best[height<=720]/bestvideo+bestaudio/best", "video"),
    "480": ("bestvideo[height<=480]+bestaudio/best[height<=480]/best", "video"),
    "mp3": ("bestaudio/best", "audio"),
    "srt": ("", "subtitles"),
}
# download lines (aria2c connections); 1 = plain yt-dlp, the gentlest for sites that block fast downloads
SPEEDS = {"native": 1, "safe2": 2, "safe4": 4, "fast8": 8, "turbo16": 16}
KEEP_BILI_QUERY = {"p", "bvid", "aid", "oid", "sid", "season_id", "ep_id", "list_id", "mid"}
_URL_IN_TEXT = re.compile(r"https?://[^\s　\"'<>，。！？；、【】（）《》「」]+", re.I)


def host_of(url: str) -> str:
    try:
        return (urlparse(url).hostname or "").lower()
    except ValueError:
        return ""


def is_short_bilibili(url: str) -> bool:
    return host_of(url) in ("b23.tv", "www.b23.tv", "bili2233.cn", "www.bili2233.cn")


def normalize_url(url: str) -> str:
    """Bilibili: the mobile / language-prefixed forms become the plain video page and only the
    query parts that pick a video are kept. Other sites are left as they are."""
    from urllib.parse import parse_qsl, urlencode, urlunparse

    try:
        u = urlparse(url)
    except ValueError:
        return url
    host = (u.hostname or "").lower()
    if not (host == "bilibili.com" or host.endswith(".bilibili.com")):
        return url
    if host in ("m.bilibili.com", "bilibili.com"):
        host = "www.bilibili.com"
    path = re.sub(r"^/(?:en|zh(?:-hans|-hant|-cn|-tw|-hk)?|ja|ko|th|vi|id|ms)(?=/(?:video|bangumi|list|medialist|cheese|festival|s/video)/)", "", u.path, flags=re.I)
    path = re.sub(r"^/s/video/", "/video/", path, flags=re.I)
    q = [(k, v) for k, v in parse_qsl(u.query, keep_blank_values=False) if k.lower() in KEEP_BILI_QUERY]
    return urlunparse((u.scheme or "https", host, path, "", urlencode(q), ""))


def extract_urls(text: str, resolve=None) -> List[str]:
    """Every address in [text] (one or many lines, share texts with words around the link),
    in order, without repeats. [resolve] turns a short Bilibili link into the real one."""
    out: List[str] = []
    for raw in (text or "").splitlines():
        line = raw.strip()
        found = _URL_IN_TEXT.findall(line)
        if not found and re.fullmatch(r"(BV[0-9A-Za-z]+|av\d+)", line):
            found = ["https://www.bilibili.com/video/" + line]
        for u in found:
            u = u.rstrip(".,;!?)'\"")
            if not u:
                continue
            if resolve is not None and is_short_bilibili(u):
                try:
                    u = resolve(u) or u
                except Exception:
                    pass
            u = normalize_url(u)
            if u not in out:
                out.append(u)
    return out


def parse_playlist_items(text: str) -> Tuple[Optional[str], str]:
    """'4-7' or '4,6,8' (yt-dlp's own syntax: also 1:3 and -5:) -> (items, error). Empty text = all parts."""
    t = re.sub(r"\s+", "", text or "")
    if not t:
        return None, ""
    if not re.fullmatch(r"[0-9,\-:]+", t):
        return None, "အပိုင်းနံပါတ်ပုံစံ မမှန်ဘူး (ဥပမာ 4-7 သို့မဟုတ် 4,6,8)"
    if len(t) > 120:
        return None, "အပိုင်းနံပါတ် စာရင်း ရှည်လွန်းတယ်"
    return t, ""


def sub_langs(mode: str, has_youtube: bool = False) -> List[str]:
    """Which subtitle languages to ask for: auto / all / zh / en / my."""
    if mode == "all":
        return ["all", "-danmaku", "-live_chat"]
    if mode == "zh":
        return ["zh-Hans", "zh-CN", "zh-Hant", "zh-TW", "zh-HK", "zh.*", "zh"]
    if mode == "en":
        return ["en.*", "en"]
    if mode == "my":
        return ["my.*", "my"]
    if has_youtube:
        return ["en", "my", "zh-Hans", "zh-CN", "zh"]  # fewer requests: YouTube answers 429 to long lists
    return ["all", "-danmaku", "-live_chat"]


def aria2_args(connections: int) -> List[str]:
    n = max(1, min(16, int(connections)))
    return ["-x", str(n), "-s", str(n), "-k", "1M", "--file-allocation=none", "--summary-interval=0"]


def download_hint(log: str, platforms=None, has_cookie: bool = False, used_aria2c: bool = False, subtitles_only: bool = False) -> str:
    """What to try next, in Burmese, for an error text from yt-dlp (several hints possible)."""
    low = (log or "").lower()
    platforms = {str(x).lower() for x in (platforms or [])}
    out: List[str] = []
    if subtitles_only and ("subtitles are only available when logged in" in low or ("bilibili" in platforms and not has_cookie)):
        out.append("Bilibili က စာတန်းထိုး ဒေါင်းဖို့ login ဝင်ထားတဲ့ cookies.txt လိုတယ်")
    elif subtitles_only and not any(k in low for k in ("403", "404", "412", "429", "bot", "region")):
        out.append("ဒီဗီဒီယိုမှာ soft subtitle မပါတာ / ရွေးထားတဲ့ ဘာသာစကား မရှိတာ ဖြစ်နိုင်တယ်")
    if "sign in to confirm" in low or "not a bot" in low or "confirm you’re not a bot" in low:
        out.append("YouTube က Colab ကို bot လို့ထင်နေတယ် — cookies.txt ကို /content/cookies.txt မှာ တင်ပါ (Notebook Cell 5b)၊ ဒါမှမဟုတ် runtime ကို Disconnect and delete လုပ်ပြီး IP အသစ်ယူပါ")
    if "412" in low or "precondition failed" in low:
        out.append("Bilibili က Colab IP ကို ခေတ္တပိတ်ထားတယ် — runtime ပြန်စ (Disconnect and delete) ပြီး IP အသစ်နဲ့ ပြန်စမ်းပါ")
    if "403" in low or "aria2c exited" in low or "forbidden" in low:
        out.append("Server က ပိတ်ထားတယ်" + (" — 'ပုံမှန် yt-dlp (1 လိုင်း)' ကို ရွေးစမ်းပါ" if used_aria2c else ""))
    if "429" in low or "too many requests" in low:
        out.append("တောင်းဆိုမှု များလွန်းလို့ ခဏတားထားတယ် — ခဏနေ ပြန်စမ်းပါ")
    if "unsupported url" in low:
        out.append("ဒီ link ပုံစံကို မဖတ်တတ်ဘူး — ဗီဒီယိုစာမျက်နှာ link ကို တိုက်ရိုက်ထည့်ပါ")
    if "private" in low or "login required" in low or "members-only" in low:
        out.append("private / login လိုတဲ့ ဗီဒီယို — အများမြင်လို့ရတဲ့ link ထည့်ပါ (သို့) cookies.txt ထည့်ပါ")
    return " · ".join(out)


# ---- subtitles in the picture (hardsub) -------------------------------------
def escape_filter_path(p: str) -> str:
    """A file path inside an ffmpeg filter argument."""
    return str(p).replace("\\", "/").replace(":", "\\:").replace("'", "\\'").replace(",", "\\,").replace("[", "\\[").replace("]", "\\]")


def hardsub_cmd(ff: str, video: str, srt: str, out: str, font: str = "Padauk", size: int = 22, gpu: bool = False) -> List[str]:
    """The subtitles burned into the picture (cannot be switched off). Needs a
    re-encode; the font must be installed (Padauk for Burmese)."""
    font = re.sub(r"[^\w \-]", "", font or "Padauk") or "Padauk"
    size = max(10, min(60, int(size)))
    style = f"FontName={font},FontSize={size},PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=1,MarginV=24"
    vf = f"subtitles='{escape_filter_path(srt)}':force_style='{style}'"
    enc = ["-c:v", "h264_nvenc", "-preset", "p4", "-cq", "23"] if gpu else ["-c:v", "libx264", "-preset", "veryfast", "-crf", "23"]
    return [ff, "-y", "-i", video, "-vf", vf] + enc + ["-c:a", "copy", "-movflags", "+faststart", out]

In [ ]:
%%writefile /content/dramaflow_backend/render_core.py
"""Turns the app's render job (the same JSON the phone's own renderer reads) into one ffmpeg command, so a long video
can be made on the Colab machine instead of the phone. Pure functions, no I/O: the caller runs ffmpeg.

Stage 1 covers: cut pieces (in order), speed, flip, whole-picture blur, blur boxes (Censor), the original sound, burned-in
subtitles, output size / fit (original, bars, crop), frame rate and bitrate. Everything else in a job makes `unsupported()`
return a reason, and the caller must refuse the job: a render must never come out missing something that was asked for."""
from typing import Dict, List, Optional, Tuple

MAX_PIECES = 400
MAX_CENSORS = 6


def _num(v, default=0.0) -> float:
    try:
        return float(v)
    except (TypeError, ValueError):
        return float(default)


def _pieces(job: dict) -> List[dict]:
    return [p for p in (job.get("pieces") or []) if isinstance(p, dict)]


def unsupported(job: dict) -> List[str]:
    """Reasons (Burmese) why this job cannot be made here yet. Empty list = it can."""
    why: List[str] = []
    pieces = _pieces(job)
    if not pieces:
        return ["ဗီဒီယို အပိုင်း မရှိဘူး"]
    if len(pieces) > MAX_PIECES:
        why.append(f"အပိုင်း {MAX_PIECES} ထက် မများရဘူး")
    last_end = -1.0
    for p in pieces:
        s, e = _num(p.get("s")), _num(p.get("e"))
        if e <= s:
            why.append("အပိုင်းတစ်ခုရဲ့ အချိန် မမှန်ဘူး")
            break
        if s < last_end:
            why.append("အပိုင်းတွေ အစီအစဉ်မကျ / ထပ်နေလို့ Colab မှာ မရသေးဘူး (မူရင်းအချိန်အတိုင်း စီပါ)")
            break
        last_end = e
    if any(p.get("f") for p in pieces):
        why.append("Freeze Frame")
    if any(p.get("c") for p in pieces):
        why.append("Crop / Zoom reframe")
    if any(_num(p.get("z")) != 0 for p in pieces):
        why.append("Zoom in / out")
    if any(p.get("l") for p in pieces):
        why.append("Filter (အရောင်)")
    if any(not (0.25 <= _num(p.get("sp", 1.0), 1.0) <= 4.0) for p in pieces):
        why.append("Speed ၀.၂၅–၄ အပြင်ဘက်")
    if any(lane for lane in (job.get("voiceLanes") or [])):
        why.append("Narration အသံ (ဖုန်းထဲက အသံဖိုင်တွေကို Colab ဆီ မပို့ရသေးဘူး)")
    if job.get("music"):
        why.append("Background music")
    if job.get("overlays"):
        why.append("Overlay (စာသား / ပုံသဏ္ဌာန် / Fade / Border …)")
    if job.get("splits"):
        why.append("Split Screen")
    if job.get("logo"):
        why.append("Logo")
    if job.get("corners"):
        why.append("ထောင့်စာသား")
    if job.get("fit") == "blur":
        why.append("Blur background (fit)")
    if len(job.get("censors") or []) > MAX_CENSORS:
        why.append(f"Blur box {MAX_CENSORS} ခုထက် များနေတယ်")
    out: List[str] = []
    for w in why:
        if w not in out:
            out.append(w)
    return out


def output_ms(job: dict) -> int:
    """Length of the finished video."""
    total = 0.0
    for p in _pieces(job):
        total += (_num(p.get("e")) - _num(p.get("s"))) / max(0.25, _num(p.get("sp", 1.0), 1.0))
    return int(total)


def atempo_chain(speed: float) -> str:
    """atempo only takes 0.5 to 2.0, so a bigger change is split in steps."""
    if abs(speed - 1.0) < 1e-6:
        return ""
    parts: List[str] = []
    s = speed
    while s > 2.0:
        parts.append("atempo=2.0")
        s /= 2.0
    while s < 0.5:
        parts.append("atempo=0.5")
        s /= 0.5
    parts.append(f"atempo={s:.6f}")
    return ",".join(parts)


def ass_color(argb: int, alpha_override: Optional[int] = None) -> str:
    """The app stores colours as 0xAARRGGBB; ASS wants &HAABBGGRR with the alpha turned round (00 = opaque)."""
    a = (argb >> 24) & 0xFF
    r, g, b = (argb >> 16) & 0xFF, (argb >> 8) & 0xFF, argb & 0xFF
    inv = 255 - a if alpha_override is None else alpha_override
    return f"&H{inv:02X}{b:02X}{g:02X}{r:02X}"


def subtitle_style(style: dict, font: str = "Padauk") -> str:
    """force_style text for ffmpeg's subtitles filter. The size is a fraction of the picture height; libass reads an
    .srt as a 288-high page, so the font size is that fraction of 288."""
    import re
    font = re.sub(r"[^\w \-]", "", font or "Padauk") or "Padauk"
    frac = min(0.12, max(0.02, _num((style or {}).get("size"), 0.05)))
    size = max(8, int(round(frac * 288)))
    color = int((style or {}).get("color", 0xFFFFFFFF)) & 0xFFFFFFFF
    box = bool((style or {}).get("bg", True))
    top = bool((style or {}).get("top", False))
    align = 6 if top else 2
    if box:
        border = "BorderStyle=3,Outline=2,Shadow=0,OutlineColour=&H99000000,BackColour=&H99000000"
    else:
        border = "BorderStyle=1,Outline=2,Shadow=1,OutlineColour=&H00000000,BackColour=&H80000000"
    return f"FontName={font},FontSize={size},PrimaryColour={ass_color(color)},{border},Alignment={align},MarginV=18"


def srt_text(cues: List[dict]) -> str:
    """The already-cut subtitles (times on the finished video) as .srt text."""
    def t(ms: float) -> str:
        ms = max(0, int(ms))
        return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"
    lines: List[str] = []
    n = 0
    for c in cues or []:
        text = str(c.get("t", "")).strip()
        s, e = _num(c.get("s")), _num(c.get("e"))
        if not text or e <= s:
            continue
        n += 1
        lines += [str(n), f"{t(s)} --> {t(e)}", text, ""]
    return "\n".join(lines)


def escape_filter_path(p: str) -> str:
    return p.replace("\\", "/").replace(":", "\\:").replace("'", "\\'")


def _f(x: float) -> str:
    return f"{x:.6f}".rstrip("0").rstrip(".") or "0"


def censor_boxes(job: dict) -> List[Tuple[float, float, float, float, float, float, float]]:
    """(left, top, width, height, start_s, end_s, sigma) per blur box, as fractions of the SOURCE picture. Times are on the
    original video, like the phone renderer reads them."""
    out = []
    aspect = _num(job.get("aspect"), 16 / 9) or 16 / 9
    for c in job.get("censors") or []:
        w = min(1.0, max(0.03, _num(c.get("w"))))
        h = min(1.0, max(0.03, w * _num(c.get("ar"), 0.5) * aspect))
        x, y = _num(c.get("x"), 0.5), _num(c.get("y"), 0.5)
        l = min(1.0, max(0.0, x - w / 2))
        r = min(1.0, max(0.0, x + w / 2))
        t = min(1.0, max(0.0, y - h / 2))
        b = min(1.0, max(0.0, y + h / 2))
        if r - l < 0.01 or b - t < 0.01:
            raise ValueError("Blur box က ပုံအပြင်ဘက် ရောက်နေတယ်")
        sigma = min(60.0, max(5.0, _num(c.get("str"), 25.0)))
        out.append((l, t, r - l, b - t, _num(c.get("s")) / 1000.0, _num(c.get("e")) / 1000.0, sigma))
    return out


def filter_graph(job: dict, has_audio: bool, srt_path: Optional[str], font: str = "Padauk") -> Tuple[str, str, Optional[str]]:
    """(filter_complex text, video label, audio label or None)."""
    pieces = _pieces(job)
    n = len(pieces)
    fps = int(_num(job.get("fps"), 30)) or 30
    out_w, out_h = int(_num(job.get("outW"))), int(_num(job.get("outH")))
    if out_w < 2 or out_h < 2:
        raise ValueError("ထုတ်မယ့် အရွယ်အစား မမှန်ဘူး")
    out_w, out_h = out_w // 2 * 2, out_h // 2 * 2
    want_audio = has_audio and _num(job.get("originalVolume")) > 0
    vol = min(2.0, max(0.0, _num(job.get("originalVolume"), 1.0)))
    g: List[str] = []

    # 1. Blur boxes, on the source picture and the source clock. The picture is split once; each box is a cut-out that is
    #    blurred and laid back over the same place, only while its time window is on.
    src = "[0:v]"
    boxes = censor_boxes(job)
    if boxes:
        names = "".join(f"[cs{i}]" for i in range(len(boxes)))
        g.append(f"[0:v]split={len(boxes) + 1}[vbase]{names}")
        cur = "[vbase]"
        for i, (l, t, w, h, a, b, sigma) in enumerate(boxes):
            g.append(f"[cs{i}]crop=w=iw*{_f(w)}:h=ih*{_f(h)}:x=iw*{_f(l)}:y=ih*{_f(t)},gblur=sigma={_f(sigma)}:steps=2[cb{i}]")
            g.append(f"{cur}[cb{i}]overlay=x=W*{_f(l)}:y=H*{_f(t)}:enable='between(t,{_f(a)},{_f(b)})'[vo{i}]")
            cur = f"[vo{i}]"
        src = cur

    # 2. One cut per piece (the same decoded picture feeds all of them).
    if n == 1:
        vsrc = [src]
    else:
        g.append(f"{src}split={n}" + "".join(f"[vs{i}]" for i in range(n)))
        vsrc = [f"[vs{i}]" for i in range(n)]
    asrc: List[str] = []
    if want_audio:
        if n == 1:
            asrc = ["[0:a]"]
        else:
            g.append(f"[0:a]asplit={n}" + "".join(f"[as{i}]" for i in range(n)))
            asrc = [f"[as{i}]" for i in range(n)]
    seg: List[str] = []
    for i, p in enumerate(pieces):
        s, e = _num(p.get("s")) / 1000.0, _num(p.get("e")) / 1000.0
        sp = _num(p.get("sp", 1.0), 1.0)
        v = f"{vsrc[i]}trim=start={_f(s)}:end={_f(e)},setpts=(PTS-STARTPTS)/{_f(sp)}"
        if p.get("fh"):
            v += ",hflip"
        if p.get("fv"):
            v += ",vflip"
        bl = _num(p.get("bl"))
        if 0.5 <= bl <= 50:
            v += f",gblur=sigma={_f(bl)}"
        v += f",fps={fps},format=yuv420p[vp{i}]"
        g.append(v)
        seg.append(f"[vp{i}]")
        if want_audio:
            a = f"{asrc[i]}atrim=start={_f(s)}:end={_f(e)},asetpts=PTS-STARTPTS"
            ch = atempo_chain(sp)
            if ch:
                a += "," + ch
            a += f",aresample=44100,volume={_f(vol)}[ap{i}]"
            g.append(a)
            seg.append(f"[ap{i}]")
    if n == 1:
        vlab = seg[0]
        alab = seg[1] if want_audio else None
    else:
        g.append("".join(seg) + f"concat=n={n}:v=1:a={1 if want_audio else 0}[vcat]" + ("[acat]" if want_audio else ""))
        vlab, alab = "[vcat]", ("[acat]" if want_audio else None)

    # 3. Size, then subtitles on the finished picture.
    fit = job.get("fit") or "orig"
    if fit == "bars":
        size = f"scale={out_w}:{out_h}:force_original_aspect_ratio=decrease,pad={out_w}:{out_h}:(ow-iw)/2:(oh-ih)/2:color=black"
    elif fit == "crop":
        size = f"scale={out_w}:{out_h}:force_original_aspect_ratio=increase,crop={out_w}:{out_h}"
    else:
        size = f"scale={out_w}:{out_h}"
    chain = size + ",setsar=1"
    if job.get("burn") and srt_path and (job.get("subtitles") or []):
        chain += f",subtitles='{escape_filter_path(srt_path)}':force_style='{subtitle_style(job.get('subStyle') or {}, font)}'"
    g.append(f"{vlab}{chain}[vout]")
    return ";\n".join(g), "[vout]", alab


def build_cmd(ffmpeg: str, src: str, out: str, job: dict, has_audio: bool, nvenc: bool,
              srt_path: Optional[str] = None, font: str = "Padauk") -> List[str]:
    why = unsupported(job)
    if why:
        raise ValueError("Colab မှာ မထုတ်နိုင်သေးတာ: " + "၊ ".join(why))
    graph, vlab, alab = filter_graph(job, has_audio, srt_path, font)
    kbps = max(300, int(_num(job.get("bitrate"), 4_000_000) / 1000))
    cmd = [ffmpeg, "-y", "-i", src, "-filter_complex", graph, "-map", vlab]
    if alab:
        cmd += ["-map", alab]
    else:
        cmd += ["-an"]
    if nvenc:
        cmd += ["-c:v", "h264_nvenc", "-preset", "p4", "-rc", "vbr", "-b:v", f"{kbps}k", "-maxrate", f"{int(kbps * 1.5)}k"]
    else:
        cmd += ["-c:v", "libx264", "-preset", "veryfast", "-b:v", f"{kbps}k", "-maxrate", f"{int(kbps * 1.5)}k", "-bufsize", f"{kbps * 2}k"]
    cmd += ["-pix_fmt", "yuv420p", "-r", str(int(_num(job.get("fps"), 30)) or 30)]
    if alab:
        cmd += ["-c:a", "aac", "-b:a", "128k"]
    cmd += ["-movflags", "+faststart", "-progress", "pipe:1", "-nostats", out]
    return cmd

In [ ]:
%%writefile /content/dramaflow_backend/nursery_core.py
"""Children's song video helpers (pure functions, no I/O): karaoke lyrics (.ass with word-by-word fill, .srt), and one
ffmpeg command that joins scene pictures / clips with the finished song. Ported from the user's own "KidsTube Pro" starter
(media.py) and tightened: word times that really add up to the line, safe paths, and one pass instead of many temp files."""
import re
from typing import Dict, List, Optional, Sequence, Tuple

MAX_ITEMS = 60
IMAGE_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
_HEADING = re.compile(r"^\s*[\[\(（【].{0,40}[\]\)）】]\s*:?\s*$")
_CJK = re.compile(r"[㐀-鿿]")


def size_for(ratio: str) -> Tuple[int, int]:
    return (720, 1280) if "9:16" in (ratio or "") else (1280, 720)


def lyric_lines(lyrics: str) -> List[str]:
    """The lines that are sung: headings such as [Chorus] and markdown marks are dropped, blank lines too."""
    out = []
    for raw in (lyrics or "").splitlines():
        t = re.sub(r"[*_`#]+", "", raw).strip()
        t = re.sub(r"^[-•]\s+", "", t)
        if not t or _HEADING.match(t):
            continue
        out.append(t)
    return out


def split_words(text: str) -> List[str]:
    """Words to fill one after another. Spaces split Latin / Burmese-with-spaces text; text with no spaces that is Chinese
    splits per character; other text with no spaces (Burmese) stays one piece, because cutting inside a syllable breaks it."""
    t = " ".join((text or "").split())
    if not t:
        return []
    if " " in t:
        return t.split(" ")
    if _CJK.search(t):
        return list(t)
    return [t]


def word_centis(words: Sequence[str], seconds: float) -> List[int]:
    """Centiseconds for each word, in proportion to its length, adding up to exactly the line's length."""
    total = max(1, int(round(max(0.0, seconds) * 100)))
    if not words:
        return []
    weights = [max(1, len(w)) for w in words]
    ws = sum(weights)
    raw = [total * w / ws for w in weights]
    out = [int(r) for r in raw]
    # hand the remaining centiseconds to the words with the biggest fractions
    left = total - sum(out)
    order = sorted(range(len(raw)), key=lambda i: raw[i] - out[i], reverse=True)
    for i in order[:max(0, left)]:
        out[i] += 1
    return out


def proportional_cues(lines: Sequence[str], total: float, pad_ratio: float = 0.06, max_pad: float = 2.0) -> List[dict]:
    """No real timing known: the song is shared by the lines in proportion to their length, with a short quiet start / end.
    Each cue is {"s", "e", "t", "words": [(word, centis)]}."""
    lines = [l for l in lines if l.strip()]
    if not lines or total <= 0:
        return []
    pad = min(max_pad, total * pad_ratio)
    usable = max(0.5, total - 2 * pad)
    weights = [max(4, len(l)) for l in lines]
    ws = sum(weights)
    cues, cur = [], pad
    for l, w in zip(lines, weights):
        d = usable * w / ws
        s, e = cur, min(total, cur + d)
        cur = e
        words = split_words(l)
        cues.append({"s": s, "e": e, "t": l, "words": list(zip(words, word_centis(words, e - s)))})
    return cues


def cues_from_segments(segments) -> List[dict]:
    """Whisper segments (start, end, text, optional words with start / end / word) into cues with word fills."""
    cues = []
    for seg in segments:
        text = (getattr(seg, "text", "") or "").strip()
        s, e = float(getattr(seg, "start", 0.0) or 0.0), float(getattr(seg, "end", 0.0) or 0.0)
        if not text or e <= s:
            continue
        ws = getattr(seg, "words", None) or []
        pairs: List[Tuple[str, int]] = []
        if ws:
            # each word fills from its own start to the next word's start, so gaps are kept inside the line
            for i, w in enumerate(ws):
                wt = (getattr(w, "word", "") or "").strip()
                if not wt:
                    continue
                ws_, we_ = float(getattr(w, "start", s) or s), float(getattr(w, "end", e) or e)
                nxt = float(getattr(ws[i + 1], "start", we_) or we_) if i + 1 < len(ws) else e
                pairs.append((wt, max(1, int(round((max(we_, nxt if nxt > ws_ else we_) - ws_) * 100)))))
        if not pairs:
            words = split_words(text)
            pairs = list(zip(words, word_centis(words, e - s)))
        cues.append({"s": s, "e": e, "t": text, "words": pairs})
    # no overlaps and at least 0.4 s
    for i, c in enumerate(cues):
        if i + 1 < len(cues) and c["e"] > cues[i + 1]["s"]:
            c["e"] = max(c["s"] + 0.1, cues[i + 1]["s"])
        if c["e"] - c["s"] < 0.4:
            c["e"] = c["s"] + 0.4
    return cues


def _ts(sec: float, comma: bool) -> str:
    sec = max(0.0, float(sec))
    if comma:
        ms = int(round(sec * 1000))
        return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"
    cs = int(round(sec * 100))
    return f"{cs // 360000:d}:{cs // 6000 % 60:02d}:{cs // 100 % 60:02d}.{cs % 100:02d}"


def srt_text(cues: Sequence[dict]) -> str:
    return "\n".join(f"{i}\n{_ts(c['s'], True)} --> {_ts(c['e'], True)}\n{c['t']}\n" for i, c in enumerate(cues, 1))


def ass_escape(t: str) -> str:
    return t.replace("\\", "/").replace("{", "(").replace("}", ")").replace("\n", " ").strip()


def ass_text(cues: Sequence[dict], width: int, height: int, font: str = "Padauk", size: Optional[int] = None) -> str:
    """Karaoke subtitles: every word fills from white to yellow as it is sung (\\kf). Colours are &HAABBGGRR."""
    size = size or max(36, int(height * 0.075))
    font = re.sub(r"[^\w \-]", "", font or "Padauk") or "Padauk"
    head = (
        "[Script Info]\nScriptType: v4.00+\n"
        f"PlayResX: {width}\nPlayResY: {height}\n\n"
        "[V4+ Styles]\n"
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, "
        "StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
        f"Style: Kids,{font},{size},&H0000E5FF,&H00FFFFFF,&H002B0A3D,&H80000000,-1,0,0,0,100,100,1,0,1,4,2,2,40,40,{int(height * 0.06)},1\n\n"
        "[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
    )
    ev = []
    for c in cues:
        words = c.get("words") or [(c["t"], max(1, int(round((c["e"] - c["s"]) * 100))))]
        line = " ".join(f"{{\\kf{cs}}}{ass_escape(w)}" for w, cs in words)
        ev.append(f"Dialogue: 0,{_ts(c['s'], False)},{_ts(c['e'], False)},Kids,,0,0,0,,{line}")
    return head + "\n".join(ev) + "\n"


def plan_durations(count: int, song: float, weights: Optional[Sequence[float]] = None, min_s: float = 1.5) -> List[float]:
    """Seconds for each of [count] scenes, adding up to the song length. [weights] (for example seconds the app planned)
    are scaled to fit; without them the scenes are equal. No scene is shorter than [min_s] when the song allows it."""
    if count <= 0 or song <= 0:
        return []
    w = [max(0.0, float(x)) for x in weights] if weights and len(weights) == count else [1.0] * count
    if sum(w) <= 0:
        w = [1.0] * count
    tot = sum(w)
    d = [song * x / tot for x in w]
    floor = min(min_s, song / count)
    short = sum(floor - x for x in d if x < floor)
    spare = sum(x - floor for x in d if x >= floor)
    if short > 0 and spare > 0:
        k = min(1.0, short / spare)
        d = [floor if x < floor else x - (x - floor) * k for x in d]
    return d


def is_image(name: str) -> bool:
    return (name or "").lower().endswith(IMAGE_EXT)


def _f(x: float) -> str:
    return f"{x:.4f}".rstrip("0").rstrip(".") or "0"


def build_slideshow_cmd(ffmpeg: str, items: Sequence[str], audio: str, out: str, durations: Sequence[float], song: float,
                        ratio: str = "16:9", fps: int = 30, sub_path: Optional[str] = None, font: str = "Padauk",
                        nvenc: bool = False, loudnorm: bool = True) -> List[str]:
    """One ffmpeg command: every picture (slow zoom, in and out in turn) or clip (looped if it is shorter than its scene)
    shown for its scene time, joined, the lyrics burned in (.ass karaoke or .srt) and the song as the sound."""
    n = len(items)
    if n < 1 or n > MAX_ITEMS:
        raise ValueError(f"ပုံ/clip 1 ကနေ {MAX_ITEMS} ထိပဲ ရတယ်")
    if len(durations) != n:
        raise ValueError("scene အချိန် အရေအတွက် မကိုက်ဘူး")
    w, h = size_for(ratio)
    fps = max(10, min(60, int(fps)))
    cmd = [ffmpeg, "-y"]
    chains = []
    for i, (src, d) in enumerate(zip(items, durations)):
        frames = max(2, int(round(d * fps)))
        if is_image(src):
            cmd += ["-i", src]
            sw, sh = (int(w * 1.25) // 2) * 2, (int(h * 1.25) // 2) * 2
            if i % 2 == 0:
                z = "min(zoom+0.0008,1.12)"
            else:
                z = "if(eq(on,0),1.12,max(zoom-0.0008,1.0))"
            vf = (f"scale={sw}:{sh}:force_original_aspect_ratio=increase,crop={sw}:{sh},"
                  f"zoompan=z='{z}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={frames}:s={w}x{h}:fps={fps},setsar=1,format=yuv420p")
        else:
            cmd += ["-stream_loop", "-1", "-t", _f(d + 0.2), "-i", src]
            vf = (f"scale={w}:{h}:force_original_aspect_ratio=increase,crop={w}:{h},fps={fps},"
                  f"trim=duration={_f(d)},setpts=PTS-STARTPTS,setsar=1,format=yuv420p")
        chains.append(f"[{i}:v]{vf}[v{i}]")
    cmd += ["-i", audio]
    cat = "".join(f"[v{i}]" for i in range(n)) + f"concat=n={n}:v=1:a=0,tpad=stop_mode=clone:stop_duration=1[vj]"
    graph = ";".join(chains + [cat])
    last = "[vj]"
    if sub_path:
        esc = sub_path.replace("\\", "/").replace(":", "\\:").replace("'", "\\'")
        if sub_path.lower().endswith(".ass"):
            graph += f";{last}ass='{esc}'[vout]"
        else:
            fname = re.sub(r"[^\w -]", "", font) or "Padauk"
            style = f"FontName={fname},FontSize=22,PrimaryColour=&H00FFFFFF,OutlineColour=&H002B0A3D,BorderStyle=1,Outline=3,Alignment=2"
            graph += f";{last}subtitles='{esc}':force_style='{style}'[vout]"
        last = "[vout]"
    cmd += ["-filter_complex", graph, "-map", last, "-map", f"{n}:a:0"]
    if loudnorm:
        cmd += ["-af", "loudnorm=I=-14:TP=-1.5:LRA=11"]
    cmd += ["-t", _f(song)]
    if nvenc:
        cmd += ["-c:v", "h264_nvenc", "-preset", "p4", "-rc", "vbr", "-cq", "23", "-b:v", "0"]
    else:
        cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "20"]
    cmd += ["-pix_fmt", "yuv420p", "-r", str(fps), "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", "-progress", "pipe:1", "-nostats", out]
    return cmd

In [ ]:
%%writefile /content/dramaflow_backend/studio_ops.py
"""Creator Studio operations (download, split, translate, subtitles, zip, YouTube).

main.py only gives these functions a request and an [Env] (where the library
folder is, how to report progress, whether the user pressed cancel / pause).
yt-dlp, ffmpeg and the Google libraries are used here, imported only when
needed, so the backend still starts without them. The logic that needs no
network is in studio_core.py.
"""
import base64
import json
import os
import re
import shutil
import subprocess
import threading
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Callable, List, Optional

import queue
import tempfile

import nursery_core as nc
import render_core as rc
import studio_core as sc
from ingest_core import check_url, classify, cookie_note, cookies_problem, cookies_source, parse_ffmpeg_duration

MAX_UPLOAD_BYTES = 8 * 1024 * 1024  # small files from the phone (subtitles, pictures)
MAX_UPLOAD_MEDIA_BYTES = 24 * 1024 * 1024  # a finished song or a short AI clip (sent as base64, so keep it small); long videos never travel this way
MIN_FREE_BYTES = 1024**3
IMAGE_EXT = (".jpg", ".jpeg", ".png", ".webp")
MIME = {".mp4": "video/mp4", ".mov": "video/quicktime", ".webm": "video/webm", ".mkv": "video/x-matroska", ".avi": "video/x-msvideo",
        ".flv": "video/x-flv", ".m4v": "video/mp4"}
KEEP_EXT = sc.VIDEO_EXT + sc.AUDIO_EXT + sc.SUB_EXT
RAW_SUB_EXT = (".vtt", ".ass", ".ssa", ".ttml", ".srv1", ".srv2", ".srv3", ".json3", ".bcc", ".xml", ".lrc")
TEMP_SUFFIX = (".part", ".ytdl", ".aria2", ".temp", ".tmp")
SLOW_SITES = {"youtube", "tiktok", "rednote", "bilibili", "douyin"}  # these block fast multi-line downloads (403): plain yt-dlp is used


class Canceled(BaseException):
    """Raised inside a download hook / loop; BaseException so yt-dlp does not swallow it."""


class Env:
    def __init__(self, lib: Path, ffmpeg: str, progress: Callable[[str, int], None], canceled: Callable[[], bool],
                 paused: Callable[[], bool] = lambda: False, detail: Callable[..., None] = lambda **k: None,
                 gemini_many: Optional[Callable[[List[str], str], List[str]]] = None):
        self.lib, self.ffmpeg, self.progress, self.canceled, self.paused, self.detail, self.gemini_many = lib, ffmpeg, progress, canceled, paused, detail, gemini_many

    def check(self):
        if self.canceled():
            raise Canceled()


def kind(name: str) -> str:
    low = (name or "").lower()
    return "image" if low.endswith(IMAGE_EXT) else sc.kind_of(name)


# ---- the library ------------------------------------------------------------
def list_library(lib: Path) -> List[dict]:
    out = []
    try:
        entries = list(lib.iterdir())
    except OSError:
        return []
    for f in entries:
        if not f.is_file() or not sc.safe_name(f.name):
            continue
        try:
            st = f.stat()
        except OSError:
            continue
        out.append({"name": f.name, "size": st.st_size, "kind": kind(f.name) if kind(f.name) != "other" else ("zip" if f.name.lower().endswith(".zip") else "other"), "mtime": int(st.st_mtime)})
    return sorted(out, key=lambda d: -d["mtime"])


def file_path(lib: Path, name: str, want=None) -> Path:
    """The path of a library file, or ValueError (Burmese) when the name is not a plain existing file."""
    if not sc.safe_name(name or ""):
        raise ValueError("ဖိုင်နာမည် မမှန်ဘူး")
    p = lib / name
    if not p.is_file():
        raise ValueError(f"'{name}' ဖိုင် မတွေ့ဘူး (Library ထဲ မရှိတော့ဘူး)")
    if want and kind(name) not in want:
        raise ValueError(f"'{name}' က {'/'.join(want)} ဖိုင် မဟုတ်ဘူး")
    return p


def save_upload(lib: Path, name: str, data_b64: str) -> str:
    """A small file from the phone (subtitles, a thumbnail) into the library. Returns the stored name."""
    if not sc.safe_name(name or "") or kind(name) not in ("srt", "image", "audio", "video"):
        raise ValueError("စာတန်း (.srt)၊ ပုံ (.jpg/.png)၊ သီချင်း (.mp3/.wav) သို့မဟုတ် clip အတိုလေး (.mp4) ဖိုင်ပဲ တင်လို့ရတယ်")
    try:
        raw = base64.b64decode(data_b64 or "", validate=False)
    except Exception:
        raise ValueError("ဖိုင်ဒေတာ မဖတ်နိုင်ဘူး")
    if not raw:
        raise ValueError("ဖိုင် အလွတ်ဖြစ်နေတယ်")
    limit = MAX_UPLOAD_MEDIA_BYTES if kind(name) in ("audio", "video") else MAX_UPLOAD_BYTES
    if len(raw) > limit:
        raise ValueError(f"ဖိုင်ကြီးလွန်းတယ် ({limit // 1048576} MB ထက် မကျော်ရ)")
    lib.mkdir(parents=True, exist_ok=True)
    stem, ext = os.path.splitext(name)
    final = sc.unique_name(sc.make_stem(stem), ext.lower(), [f.name for f in lib.iterdir()])
    (lib / final).write_bytes(raw)
    return final


def delete_file(lib: Path, name: str):
    file_path(lib, name).unlink()


def free_ok(lib: Path):
    try:
        if shutil.disk_usage(lib).free < MIN_FREE_BYTES:
            raise ValueError("Backend ရဲ့ disk နေရာ မလောက်တော့ဘူး (၁ GB အောက်) — Library ထဲက ဖိုင်တွေ ဖျက်ပါ")
    except OSError:
        pass


def has_nvenc(ffmpeg: str) -> bool:
    try:
        if "h264_nvenc" not in subprocess.run([ffmpeg, "-hide_banner", "-encoders"], capture_output=True, text=True, timeout=10).stdout:
            return False
        return subprocess.run(["nvidia-smi"], capture_output=True, timeout=10).returncode == 0
    except Exception:
        return False


# ---- running ffmpeg ---------------------------------------------------------
def run_ffmpeg(env: Env, cmd: List[str], timeout: int = 6 * 3600):
    """ffmpeg that stops when the user cancels. Raises RuntimeError with the end of ffmpeg's message."""
    cmd = [cmd[0], "-hide_banner", "-loglevel", "error"] + cmd[1:]
    p = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    started = time.time()
    err = ""
    while True:
        try:
            _, err = p.communicate(timeout=0.5)
            break
        except subprocess.TimeoutExpired:
            if env.canceled():
                p.kill()
                p.communicate()
                raise Canceled()
            if time.time() - started > timeout:
                p.kill()
                p.communicate()
                raise RuntimeError("ffmpeg အချိန်ကြာလွန်းလို့ ရပ်လိုက်တယ်")
    if p.returncode != 0:
        raise RuntimeError("ffmpeg မအောင်မြင်ဘူး: " + (err or "").strip()[-300:])


def duration_ms(env: Env, path: Path) -> int:
    try:
        p = subprocess.run([env.ffmpeg, "-i", str(path)], capture_output=True, text=True, timeout=60)
        return parse_ffmpeg_duration(p.stderr)
    except Exception:
        return 0


# ---- download ---------------------------------------------------------------
def _resolve_short(url: str) -> str:
    ok, _ = check_url(url)
    if not ok:
        return url
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/131.0.0.0"})
        with urllib.request.urlopen(req, timeout=8) as r:
            return r.geturl() or url
    except Exception:
        return url


_NAME_RE = re.compile(r"^(?P<title>.*) \[(?P<id>[^\]]+)\](?:\.(?P<lang>[\w\-]+))?\.(?P<ext>[A-Za-z0-9]+)$")


def final_name(raw_name: str, existing) -> str:
    """'<title> [id].mp4' / '<title> [id].zh-Hans.srt' -> a clean library name."""
    m = _NAME_RE.match(raw_name)
    if m:
        stem, ext, lang = sc.make_stem(m.group("title")), "." + m.group("ext").lower(), m.group("lang")
        if lang and ext == ".srt":
            return sc.unique_name(stem + "." + lang, ext, existing)
        return sc.unique_name(stem, ext, existing)
    base, ext = os.path.splitext(raw_name)
    return sc.unique_name(sc.make_stem(base), ext.lower(), existing)


def op_download(env: Env, req) -> dict:
    try:
        import yt_dlp
    except Exception:
        raise RuntimeError("yt-dlp မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ)")
    quality = req.quality if req.quality in sc.QUALITIES else "720"
    fmt, what = sc.QUALITIES[quality]
    urls = sc.extract_urls(req.url, resolve=_resolve_short)
    if not urls:
        raise ValueError("Link မတွေ့ဘူး (http:// သို့မဟုတ် https:// နဲ့ စတဲ့ link ထည့်ပါ)")
    if len(urls) > 30:
        raise ValueError("Link 30 ခုထက် မများရ")
    for u in urls:
        ok, why = check_url(u)
        if not ok:
            raise ValueError(f"{u[:60]} — {why}")
    items, err = sc.parse_playlist_items(getattr(req, "playlist_items", ""))
    if err:
        raise ValueError(err)
    free_ok(env.lib)
    env.lib.mkdir(parents=True, exist_ok=True)
    kinds = {classify(u) for u in urls}
    conns = sc.SPEEDS.get(getattr(req, "speed", "safe4"), 4)
    use_aria2c = bool(shutil.which("aria2c")) and conns > 1 and what == "video" and not (kinds & SLOW_SITES)
    langs = sc.sub_langs(getattr(req, "sub_langs", "auto"), "youtube" in kinds)
    want_subs = what == "subtitles" or bool(getattr(req, "subtitles", False))
    result = {"files": [], "notes": [], "errors": []}
    src = cookies_source()
    cookie_text = ""
    if src is not None:
        cookie_text = src.read_text(encoding="utf-8", errors="ignore")
        bad = cookies_problem(cookie_text)
        if bad:
            raise ValueError(bad)
        result["notes"].append("cookies.txt သုံးထားတယ်")
        for u in urls:
            note = cookie_note(cookie_text, classify(u))
            if note:
                result["notes"].append(note)
    if use_aria2c:
        result["notes"].append(f"aria2c {conns} လိုင်းနဲ့ ဆွဲတယ်")
    elif conns > 1 and what == "video" and (kinds & SLOW_SITES):
        result["notes"].append("YouTube/TikTok/Red Note/Bilibili မှာ 403 မဖြစ်အောင် ပုံမှန် yt-dlp (1 လိုင်း) နဲ့ ဆွဲတယ်")

    for ui, url in enumerate(urls, 1):
        env.check()
        tmp = env.lib / f".tmp_{int(time.time())}_{ui}"
        tmp.mkdir(parents=True, exist_ok=True)
        log: List[str] = []

        class Quiet:
            def debug(self, m):
                pass

            def info(self, m):
                pass

            def warning(self, m):
                log.append(str(m))

            def error(self, m):
                log.append(str(m))

        def hook(d, ui=ui):
            while env.paused():
                env.check()
                env.detail(paused=True)
                time.sleep(0.5)
            env.check()
            if d.get("status") == "downloading":
                tot = d.get("total_bytes") or d.get("total_bytes_estimate") or 0
                got = d.get("downloaded_bytes") or 0
                pct = int(100 * got / tot) if tot else 0
                env.detail(paused=False, downloaded_mb=round(got / 1048576, 1), total_mb=round(tot / 1048576, 1), speed_mbs=round((d.get("speed") or 0) / 1048576, 2),
                           eta=d.get("eta"), file=os.path.basename(d.get("filename") or ""), fragment=(f"{d.get('fragment_index')}/{d.get('fragment_count')}" if d.get("fragment_count") else ""))
                env.progress(f"ဆွဲနေတယ် ({ui}/{len(urls)}) {pct}%" if tot else f"ဆွဲနေတယ် ({ui}/{len(urls)})", min(95, int(100 * (ui - 1 + (pct / 100 if tot else 0.0)) / len(urls))))

        opts = {
            "outtmpl": str(tmp / "%(title).80s [%(id)s].%(ext)s"),
            "windowsfilenames": True,
            "noplaylist": not bool(getattr(req, "playlist", False)),
            "quiet": True,
            "no_warnings": True,
            "logger": Quiet(),
            "socket_timeout": 30,
            "retries": 5,
            "fragment_retries": 5,
            "ffmpeg_location": env.ffmpeg,
            "progress_hooks": [hook],
            "ignoreerrors": False,
        }
        if items:
            opts["playlist_items"] = items
        if what == "video":
            opts["format"] = fmt
            opts["merge_output_format"] = "mp4"
        elif what == "audio":
            opts["format"] = fmt
            opts["postprocessors"] = [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}]
        else:
            opts["skip_download"] = True
        if want_subs:
            opts.update({"writesubtitles": True, "subtitleslangs": langs, "subtitlesformat": "srt/best"})
            if what == "subtitles" and getattr(req, "auto_subs", False):
                opts["writeautomaticsub"] = True
            opts.setdefault("postprocessors", []).append({"key": "FFmpegSubtitlesConvertor", "format": "srt", "when": "before_dl"})
        if use_aria2c:
            opts["external_downloader"] = {"default": "aria2c"}
            opts["external_downloader_args"] = {"aria2c": sc.aria2_args(conns)}
        if src is not None:
            ck = tmp / "cookies.txt"  # a private copy: yt-dlp writes to its cookie file
            ck.write_text(cookie_text, encoding="utf-8")
            try:
                os.chmod(ck, 0o600)
            except OSError:
                pass
            opts["cookiefile"] = str(ck)
        try:
            env.progress(f"ဆွဲမယ် ({ui}/{len(urls)})", int(100 * (ui - 1) / len(urls)) + 1)
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.extract_info(url, download=True)
            made = 0
            for f in sorted(tmp.iterdir()):
                if f.name == "cookies.txt" or f.name.endswith(TEMP_SUFFIX) or not f.is_file():
                    continue
                low = f.name.lower()
                if low.endswith(RAW_SUB_EXT) or re.search(r"\.(danmaku|live_chat)\.", low):
                    continue
                if not low.endswith(KEEP_EXT) or re.search(r"\.f\d+\.", low):
                    continue
                dest = final_name(f.name, [x.name for x in env.lib.iterdir()])
                shutil.move(str(f), str(env.lib / dest))
                result["files"].append(dest)
                made += 1
            if made == 0:
                hint = sc.download_hint("\n".join(log), kinds, src is not None, use_aria2c, what == "subtitles")
                raise RuntimeError(hint or "ဖိုင်တစ်ခုမှ မရဘူး (link မှန်လား၊ ဒီ website အတွက် yt-dlp အသစ်လိုနိုင်တယ်)")
        except Canceled:
            raise
        except Exception as e:  # one bad link does not stop the others
            msg = str(e)
            hint = sc.download_hint(msg + "\n" + "\n".join(log), kinds, src is not None, use_aria2c, what == "subtitles")
            result["errors"].append(f"{url[:70]}: " + ((hint + " · ") if hint and hint not in msg else "") + msg[:240])
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
    if not result["files"]:
        raise RuntimeError(" | ".join(result["errors"]) or "ဖိုင်တစ်ခုမှ မရဘူး")
    env.progress("ပြီးပြီ", 100)
    return result


# ---- split ------------------------------------------------------------------
def op_split(env: Env, req) -> dict:
    src = file_path(env.lib, req.file, ("video",))
    total = duration_ms(env, src) / 1000
    if total < sc.MIN_PART_SECONDS:
        raise ValueError("ဗီဒီယိုအရှည်ကို မဖတ်နိုင်ဘူး (ဖိုင်ပျက်နေနိုင်တယ်)")
    if (getattr(req, "split_mode", "count") or "count") == "ranges":
        parts, err = sc.parse_ranges(req.ranges, total)
        if err:
            raise ValueError(err)
    else:
        parts = sc.equal_parts(total, int(getattr(req, "count", 2) or 2))
        if not parts:
            raise ValueError("ဗီဒီယို တိုလွန်းလို့ မခွဲနိုင်ဘူး")
    cues = []
    if getattr(req, "srt_file", ""):
        cues = sc.parse_srt(file_path(env.lib, req.srt_file, ("srt",)).read_text(encoding="utf-8", errors="ignore"))
    free_ok(env.lib)
    stem = os.path.splitext(src.name)[0]
    ext = src.suffix.lower()
    files, notes = [], []
    for i, (a, b) in enumerate(parts, 1):
        env.check()
        env.progress(f"အပိုင်း {i}/{len(parts)} ဖြတ်နေတယ်", int(100 * (i - 1) / len(parts)) + 1)
        name = sc.unique_name(f"{stem}_Part{i:02d}", ext, [f.name for f in env.lib.iterdir()])
        out = env.lib / name
        try:
            run_ffmpeg(env, sc.split_cmd(env.ffmpeg, str(src), a, b - a, str(out), bool(getattr(req, "reencode", False))))
        except BaseException:
            out.unlink(missing_ok=True)
            raise
        if not out.exists() or out.stat().st_size < 1024:
            out.unlink(missing_ok=True)
            raise RuntimeError(f"အပိုင်း {i} မထွက်ဘူး")
        files.append(name)
        if cues:
            part_cues = sc.slice_srt(cues, a, b)
            if part_cues:
                sname = sc.unique_name(f"{stem}_Part{i:02d}", ".srt", [f.name for f in env.lib.iterdir()])
                (env.lib / sname).write_text(sc.serialize_srt(part_cues), encoding="utf-8")
                files.append(sname)
    if not getattr(req, "reencode", False):
        notes.append("Stream copy နဲ့ ဖြတ်တာမို့ အနီးဆုံး key frame မှာ ဖြတ်တယ် (အတိအကျ လိုရင် 'Re-encode' ဖွင့်ပါ)")
    env.progress("ပြီးပြီ", 100)
    return {"files": files, "notes": notes, "errors": []}


# ---- subtitles --------------------------------------------------------------
def _free_translator(src_lang: str):
    from deep_translator import GoogleTranslator

    tr = GoogleTranslator(source=src_lang or "auto", target="my")
    sep = "\n<<<SRT_SEP>>>\n"

    def many(texts: List[str]) -> List[str]:
        if len(texts) == 1:
            return [tr.translate(texts[0]) or ""]
        res = tr.translate(sep.join(t.replace("<<<SRT_SEP>>>", " ") for t in texts)) or ""
        parts = re.split(r"\s*<<<\s*SRT_SEP\s*>>>\s*", res)
        time.sleep(0.25)
        return parts

    return many


def op_translate(env: Env, req) -> dict:
    src = file_path(env.lib, req.file, ("srt",))
    cues = sc.parse_srt(src.read_text(encoding="utf-8", errors="ignore"))
    if not cues:
        raise ValueError("SRT ထဲမှာ စာတန်းစာကြောင်း မတွေ့ဘူး")
    engine = getattr(req, "engine", "free") or "free"
    lang = getattr(req, "src_lang", "auto") or "auto"
    if engine == "gemini":
        if env.gemini_many is None:
            raise ValueError("Gemini key မပါဘူး (Settings မှာ ထည့်ပါ)")
        many = lambda texts: env.gemini_many(texts, lang)  # noqa: E731
    else:
        try:
            many = _free_translator(lang)
        except Exception:
            raise RuntimeError("deep-translator မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ) — ဒါမှမဟုတ် Gemini engine ရွေးပါ")

    def prog(done, total):
        env.check()
        env.progress(f"ဘာသာပြန်နေတယ် {done}/{total}", int(100 * done / max(1, total)))

    new, failed = sc.translate_cues(cues, many, prog)
    if failed >= len(cues):
        raise RuntimeError("ဘာသာပြန် တစ်ကြောင်းမှ မရဘူး (engine ကန့်သတ်ထားတာ/key မမှန်တာ ဖြစ်နိုင်တယ် — ခဏနေ ပြန်စမ်း၊ ဒါမှမဟုတ် တခြား engine ရွေးပါ)")
    stem = re.sub(r"_my$", "", os.path.splitext(src.name)[0])
    out = sc.unique_name(stem + "_my", ".srt", [f.name for f in env.lib.iterdir()])
    (env.lib / out).write_text(sc.serialize_srt(new), encoding="utf-8")
    notes = [f"စာကြောင်း {len(cues) - failed}/{len(cues)} ဘာသာပြန်ပြီး"]
    if failed:
        notes.append(f"{failed} ကြောင်း မပြန်နိုင်လို့ မူရင်းအတိုင်း ထားတယ်")
    env.progress("ပြီးပြီ", 100)
    return {"files": [out], "notes": notes, "errors": []}


def _load_whisper(model_name: str):
    """(model, device note). Uses the GPU when there is one, and falls back to the CPU if the GPU cannot be started."""
    from faster_whisper import WhisperModel

    cuda = False
    try:
        import ctranslate2

        cuda = ctranslate2.get_cuda_device_count() > 0
    except Exception:
        cuda = False
    if cuda:
        try:
            return WhisperModel(model_name, device="cuda", compute_type="float16"), "GPU"
        except Exception:
            pass
    return WhisperModel(model_name, device="cpu", compute_type="int8"), "CPU"


def op_transcribe(env: Env, req) -> dict:
    """Speech to text with Whisper (faster-whisper), on the Colab machine: no paid service. The result is an .srt in the library
    (the video / audio is only read), which can then be translated to Burmese or put into the video."""
    src = file_path(env.lib, req.file, ("video", "audio"))
    model_name = sc.whisper_model_name(getattr(req, "whisper_model", "small"))
    lang = sc.whisper_lang(getattr(req, "src_lang", "auto"))
    env.progress(f"Whisper model ({model_name}) ပြင်နေတယ် — ပထမဆုံးအကြိမ် ဒေါင်းရလို့ ကြာနိုင်တယ်", 2)
    try:
        model, device = _load_whisper(model_name)
    except ImportError:
        raise RuntimeError("faster-whisper မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ)")
    env.check()
    started = time.time()
    segments, info = model.transcribe(str(src), language=lang, vad_filter=bool(getattr(req, "whisper_vad", True)), beam_size=5)
    total = float(getattr(info, "duration", 0.0) or 0.0)
    found = []
    for seg in segments:  # a generator: the work happens here
        env.check()
        found.append(seg)
        end = float(getattr(seg, "end", 0.0) or 0.0)
        if total > 0:
            env.progress(f"စာတန်းထုတ်နေတယ် {int(end) // 60}:{int(end) % 60:02d} / {int(total) // 60}:{int(total) % 60:02d}", 3 + int(95 * min(end, total) / total))
    cues = sc.segments_to_cues(found)
    if not cues:
        raise RuntimeError("စကားသံ တစ်ခုမှ မတွေ့ဘူး (အသံမပါတာ / ဇာတ်ကြောင်းက တီးလုံးသက်သက်ဖြစ်တာ ဖြစ်နိုင်တယ်)")
    spoken = getattr(info, "language", None) or lang or "auto"
    stem = os.path.splitext(src.name)[0]
    out = sc.unique_name(f"{stem}_{spoken}", ".srt", [f.name for f in env.lib.iterdir()])
    (env.lib / out).write_text(sc.serialize_srt(cues), encoding="utf-8")
    took = max(1, int(time.time() - started))
    notes = [f"စာကြောင်း {len(cues)} ခု · ဘာသာ {spoken} · Whisper {model_name} ({device}) · {took} စက္ကန့်"]
    prob = getattr(info, "language_probability", None)
    if lang is None and isinstance(prob, (int, float)) and prob < 0.6:
        notes.append(f"ဘာသာကို အလိုအလျောက် ခန့်မှန်းတာ ယုံကြည်မှု နည်းတယ် ({int(prob * 100)}%) — ဘာသာကို ကိုယ်တိုင် ရွေးပြီး ပြန်လုပ်ကြည့်ပါ")
    notes.append("စကားသံကနေ ထုတ်တာမို့ အမှားပါနိုင်တယ်၊ Subtitle Studio မှာ ပြင်ပါ")
    env.progress("ပြီးပြီ", 100)
    return {"files": [out], "notes": notes, "errors": []}


def op_subtitle_video(env: Env, req) -> dict:
    """Soft CC (a track you can switch off, what YouTube also reads) and / or hardsub (burned in)."""
    video = file_path(env.lib, req.file, ("video",))
    srt = file_path(env.lib, req.srt_file, ("srt",))
    mode = getattr(req, "sub_mode", "soft") or "soft"
    if mode not in ("soft", "hard", "both"):
        raise ValueError("sub_mode မမှန်ဘူး")
    free_ok(env.lib)
    stem = os.path.splitext(video.name)[0]
    files, notes = [], []
    steps = [m for m in ("soft", "hard") if mode in (m, "both")]
    for i, m in enumerate(steps):
        env.check()
        env.progress("Soft CC ထည့်နေတယ်" if m == "soft" else "Hardsub ဖော်နေတယ် (ကြာတယ်)", int(100 * i / len(steps)) + 1)
        name = sc.unique_name(stem + ("_cc" if m == "soft" else "_hardsub"), ".mp4", [f.name for f in env.lib.iterdir()])
        out = env.lib / name
        if m == "soft":
            cmd = sc.mux_cc_cmd(env.ffmpeg, str(video), str(srt), str(out))
        else:
            gpu = has_nvenc(env.ffmpeg)
            cmd = sc.hardsub_cmd(env.ffmpeg, str(video), str(srt), str(out), getattr(req, "font", "Padauk"), int(getattr(req, "font_size", 22) or 22), gpu)
            notes.append("GPU (nvenc) နဲ့ ဖော်တယ်" if gpu else "CPU နဲ့ ဖော်တယ် (GPU runtime ရွေးရင် ပိုမြန်တယ်)")
        try:
            run_ffmpeg(env, cmd)
        except BaseException:
            out.unlink(missing_ok=True)
            raise
        files.append(name)
    env.progress("ပြီးပြီ", 100)
    return {"files": files, "notes": notes, "errors": []}


# ---- render (the phone's render job, made on this machine) ------------------
def has_audio_stream(env: Env, path: Path) -> bool:
    try:
        p = subprocess.run([env.ffmpeg, "-i", str(path)], capture_output=True, text=True, timeout=60)
        return "Audio:" in p.stderr
    except Exception:
        return False


def run_ffmpeg_progress(env: Env, cmd: List[str], total_ms: int, label: str, timeout: int = 12 * 3600):
    """ffmpeg with `-progress pipe:1`: the position it reports becomes a percentage, and the user can cancel."""
    full = [cmd[0], "-hide_banner", "-loglevel", "error"] + cmd[1:]
    p = subprocess.Popen(full, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines: "queue.Queue[str]" = queue.Queue()

    def pump():
        for ln in p.stdout:  # type: ignore[union-attr]
            lines.put(ln.rstrip("\n"))
        lines.put("")

    threading.Thread(target=pump, daemon=True).start()
    started = time.time()
    tail: List[str] = []
    done = False
    try:
        _watch_ffmpeg(env, p, lines, total_ms, label, timeout, tail, started)
    finally:
        try:
            p.stdout.close()  # type: ignore[union-attr]
        except Exception:
            pass
    if p.returncode != 0:
        raise RuntimeError("ffmpeg မအောင်မြင်ဘူး: " + " ".join(tail)[-300:])


def _watch_ffmpeg(env: Env, p, lines, total_ms: int, label: str, timeout: int, tail: List[str], started: float):
    done = False
    while not done:
        try:
            ln = lines.get(timeout=0.5)
            if ln == "" and p.poll() is not None:
                done = True
            elif ln.startswith("out_time_us=") or ln.startswith("out_time_ms="):
                try:
                    us = int(ln.split("=", 1)[1])  # both spellings are in microseconds
                    if total_ms > 0 and us > 0:
                        env.progress(label, max(1, min(99, int(100 * (us / 1000.0) / total_ms))))
                except ValueError:
                    pass
            elif ln and "=" not in ln.split(" ")[0]:
                tail.append(ln)
                tail = tail[-6:]
        except queue.Empty:
            pass
        if env.canceled():
            p.kill()
            p.wait()
            raise Canceled()
        if time.time() - started > timeout:
            p.kill()
            p.wait()
            raise RuntimeError("ffmpeg အချိန်ကြာလွန်းလို့ ရပ်လိုက်တယ်")
        if p.poll() is not None and lines.empty():
            done = True


def op_render(env: Env, req) -> dict:
    """Makes the video described by the app's render job (cut pieces, speed, Censor boxes, subtitles, size) from a video that
    is already in the library. The original file is only read. Anything the job asks for that is not supported yet makes the
    whole job refuse, so a video never comes out with something silently missing."""
    job = getattr(req, "job", None)
    if not isinstance(job, dict) or not job:
        raise ValueError("render job မပါဘူး")
    src = file_path(env.lib, req.file, ("video",))
    why = rc.unsupported(job)
    if why:
        raise ValueError("Colab မှာ မထုတ်နိုင်သေးတာ: " + "၊ ".join(why))
    free_ok(env.lib)
    total = rc.output_ms(job)
    stem = re.sub(r"[^\w\-. ]", "_", str(job.get("name") or os.path.splitext(src.name)[0])).strip(" .") or "render"
    name = sc.unique_name(stem, ".mp4", [f.name for f in env.lib.iterdir()])
    out = env.lib / name
    gpu = has_nvenc(env.ffmpeg)
    notes = ["GPU (nvenc) နဲ့ ထုတ်တယ်" if gpu else "CPU နဲ့ ထုတ်တယ် (GPU runtime ရွေးရင် ပိုမြန်တယ်)"]
    with tempfile.TemporaryDirectory() as tmp:
        srt_path = None
        if job.get("burn") and job.get("subtitles"):
            srt_path = os.path.join(tmp, "subs.srt")
            Path(srt_path).write_text(rc.srt_text(job["subtitles"]), encoding="utf-8")
        cmd = rc.build_cmd(env.ffmpeg, str(src), str(out), job, has_audio_stream(env, src), gpu, srt_path, getattr(req, "font", "Padauk") or "Padauk")
        env.progress("ထုတ်နေတယ်", 1)
        try:
            run_ffmpeg_progress(env, cmd, total, "ထုတ်နေတယ်")
        except BaseException:
            out.unlink(missing_ok=True)
            raise
    if not out.exists() or out.stat().st_size < 1024:
        out.unlink(missing_ok=True)
        raise RuntimeError("ဗီဒီယို မထွက်ဘူး")
    if job.get("censors"):
        notes.append(f"Blur box {len(job['censors'])} ခု ပါတယ် — ထုတ်ပြီးရင် မှုန်သွားတာကို ကိုယ်တိုင် ပြန်စစ်ပါ (အရင်းအမြစ်ပုံ အတိုင်း မှုန်ဖို့ ရည်ရွယ်ပေမယ့် ဖုန်း render နဲ့ အတိအကျ မတူနိုင်ဘူး)")
    env.progress("ပြီးပြီ", 100)
    return {"files": [name], "notes": notes, "errors": []}


# ---- children's song video (karaoke lyrics + pictures / clips + the finished song) ---------------------------------
def _lib_sub(env: Env, name: str) -> Path:
    """A subtitle file in the library: .srt or .ass (the karaoke file this backend writes)."""
    if not sc.safe_name(name or "") or not name.lower().endswith((".srt", ".ass")):
        raise ValueError("စာတန်းဖိုင် (.ass / .srt) မဟုတ်ဘူး")
    p = env.lib / name
    if not p.is_file():
        raise ValueError(f"'{name}' ဖိုင် မတွေ့ဘူး (Library ထဲ မရှိတော့ဘူး)")
    return p


def op_karaoke(env: Env, req) -> dict:
    """Lyrics timed to the song, as .ass (each word fills as it is sung) and .srt, written into the library.
    mode "lyrics": the person's own lyrics shared over the song by length (works for any language, only roughly on the beat).
    mode "whisper": Whisper listens to the song and gives line and word times, with the text it heard (good for English /
    Chinese, poor for sung Burmese; check the text)."""
    audio = file_path(env.lib, req.file, ("audio", "video"))
    mode = "whisper" if getattr(req, "karaoke_mode", "lyrics") == "whisper" else "lyrics"
    lines = nc.lyric_lines(getattr(req, "lyrics", ""))
    if mode == "lyrics" and not lines:
        raise ValueError("သီချင်းစာသား မပါဘူး (Lyrics ကို paste လုပ်ပါ)")
    env.progress("သီချင်းအရှည် ကြည့်နေတယ်", 3)
    total = duration_ms(env, audio) / 1000.0
    if total <= 0:
        raise RuntimeError("သီချင်းအရှည် မဖတ်နိုင်ဘူး")
    w, h = nc.size_for(getattr(req, "ratio", "16:9"))
    notes: List[str] = []
    cues: List[dict] = []
    if mode == "whisper":
        env.progress("Whisper ပြင်နေတယ် — ပထမဆုံးအကြိမ် model ဒေါင်းရလို့ ကြာနိုင်တယ်", 8)
        try:
            model, device = _load_whisper(sc.whisper_model_name(getattr(req, "whisper_model", "small")))
        except ImportError:
            raise RuntimeError("faster-whisper မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ)")
        env.check()
        segs, _info = model.transcribe(str(audio), language=sc.whisper_lang(getattr(req, "src_lang", "auto")), word_timestamps=True, vad_filter=True, beam_size=5)
        found = []
        for seg in segs:
            env.check()
            found.append(seg)
            env.progress("သီချင်းကို နားထောင်နေတယ်", 10 + int(80 * min(float(seg.end or 0), total) / total))
        cues = nc.cues_from_segments(found)
        if cues:
            notes.append(f"Whisper ({device}) နဲ့ စာကြောင်း {len(cues)} ခု — ကြားရတဲ့အတိုင်း ရေးတာမို့ စာလုံးအမှားပါနိုင်တယ်၊ သီချင်းဆိုထားရင် ပိုမှားတယ်")
        else:
            notes.append("Whisper က စာတစ်ကြောင်းမှ မရဘူး — ပေးထားတဲ့ lyrics ကိုပဲ အရှည်အလိုက် ခွဲထားတယ်")
    if not cues:
        if not lines:
            raise RuntimeError("စာကြောင်း မရဘူး၊ Lyrics ကိုလည်း မပေးထားဘူး")
        env.progress("Lyrics ကို သီချင်းအရှည်အတိုင်း ခွဲနေတယ်", 60)
        cues = nc.proportional_cues(lines, total)
        notes.append("Lyrics ကို စာကြောင်းအရှည် အလိုက် ခွဲထားတာ — အကြမ်းချိန်ပဲ၊ သီချင်းနဲ့ အတိအကျ မကိုက်နိုင်ဘူး။ ကိုက်မှု လိုရင် ထုတ်ပြီး ကြည့်ပြီး .srt ကို Subtitle Studio မှာ ပြင်ပါ")
    stem = os.path.splitext(audio.name)[0]
    existing = [f.name for f in env.lib.iterdir()]
    ass_name = sc.unique_name(f"{stem}_karaoke", ".ass", existing)
    srt_name = sc.unique_name(f"{stem}_karaoke", ".srt", existing + [ass_name])
    (env.lib / ass_name).write_text(nc.ass_text(cues, w, h, getattr(req, "font", "Padauk") or "Padauk"), encoding="utf-8")
    (env.lib / srt_name).write_text(nc.srt_text(cues), encoding="utf-8")
    env.progress("ပြီးပြီ", 100)
    return {"files": [ass_name, srt_name], "notes": notes, "errors": []}


def op_nursery(env: Env, req) -> dict:
    """The song video: the pictures / clips in `files` (in order) shown over the song in `file`, with karaoke lyrics from
    `srt_file` (an .ass or .srt made by the karaoke op) when given. `durations` are the app's planned seconds per scene
    (scaled to the song length); without them the scenes are equal. Everything is only read; the video is a new file."""
    names = list(getattr(req, "files", []) or [])
    if not names:
        raise ValueError("ပုံ / clip မရွေးထားဘူး")
    if len(names) > nc.MAX_ITEMS:
        raise ValueError(f"ပုံ/clip {nc.MAX_ITEMS} ခုထက် မများရဘူး")
    items = [str(file_path(env.lib, n, ("image", "video"))) for n in names]
    audio = file_path(env.lib, req.file, ("audio",))
    sub = _lib_sub(env, req.srt_file) if getattr(req, "srt_file", "") else None
    free_ok(env.lib)
    song = duration_ms(env, audio) / 1000.0
    if song <= 0:
        raise RuntimeError("သီချင်းအရှည် မဖတ်နိုင်ဘူး")
    weights = [float(x) for x in (getattr(req, "durations", []) or [])]
    durs = nc.plan_durations(len(items), song, weights if len(weights) == len(items) else None)
    stem = re.sub(r"[^\w\-. ]", "_", os.path.splitext(audio.name)[0]).strip(" .") or "song"
    name = sc.unique_name(f"{stem}_video", ".mp4", [f.name for f in env.lib.iterdir()])
    out = env.lib / name
    gpu = has_nvenc(env.ffmpeg)
    notes = ["GPU (nvenc) နဲ့ ထုတ်တယ်" if gpu else "CPU နဲ့ ထုတ်တယ် (GPU runtime ရွေးရင် ပိုမြန်တယ်)"]
    if len(weights) not in (0, len(items)):
        notes.append("scene အချိန် အရေအတွက် ပုံအရေအတွက်နဲ့ မကိုက်လို့ အညီအမျှ ခွဲထားတယ်")
    with tempfile.TemporaryDirectory() as tmp:
        sub_path = None
        if sub is not None:
            sub_path = os.path.join(tmp, "lyrics" + sub.suffix.lower())
            shutil.copyfile(sub, sub_path)
        cmd = nc.build_slideshow_cmd(env.ffmpeg, items, str(audio), str(out), durs, song, getattr(req, "ratio", "16:9"), 30, sub_path,
                                     getattr(req, "font", "Padauk") or "Padauk", gpu)
        env.progress("ပုံတွေနဲ့ သီချင်း ပေါင်းနေတယ်", 1)
        try:
            run_ffmpeg_progress(env, cmd, int(song * 1000), "ဗီဒီယို ထုတ်နေတယ်")
        except BaseException:
            out.unlink(missing_ok=True)
            raise
    if not out.exists() or out.stat().st_size < 1024:
        out.unlink(missing_ok=True)
        raise RuntimeError("ဗီဒီယို မထွက်ဘူး")
    notes.append("ပုံ/clip ကို သီချင်းအချိန်ပေါ် ခွဲပြထားတာ — ပုံတွေ မူပိုင်ခွင့်၊ AI ဖော်ပြချက်ကို upload မတင်ခင် စစ်ပါ")
    env.progress("ပြီးပြီ", 100)
    return {"files": [name], "notes": notes, "errors": []}


# ---- zip --------------------------------------------------------------------
def op_zip(env: Env, req) -> dict:
    names = list(getattr(req, "files", []) or [])
    if not names:
        raise ValueError("ZIP လုပ်မယ့် ဖိုင် မရွေးရသေးဘူး")
    paths = [file_path(env.lib, n) for n in names]
    free_ok(env.lib)
    base = sc.make_stem(getattr(req, "zip_name", "") or "", "studio_" + time.strftime("%Y%m%d_%H%M"))
    out = sc.unique_name(base, ".zip", [f.name for f in env.lib.iterdir()])
    with zipfile.ZipFile(env.lib / out, "w", zipfile.ZIP_STORED, allowZip64=True) as z:  # videos do not shrink: no time wasted
        for i, p in enumerate(paths, 1):
            env.check()
            env.progress(f"ZIP {i}/{len(paths)}", int(100 * i / len(paths)))
            z.write(p, arcname=p.name)
    return {"files": [out], "notes": [], "errors": []}


# ---- YouTube ----------------------------------------------------------------
YT_SCOPES = ["https://www.googleapis.com/auth/youtube.upload", "https://www.googleapis.com/auth/youtube.force-ssl"]
_YT = {"flow": None}
_YT_LOCK = threading.Lock()


def yt_secret_path() -> Optional[Path]:
    for c in (os.environ.get("DF_YT_CLIENT_SECRET"), "/content/client_secret.json", str(Path(__file__).resolve().parent / "client_secret.json")):
        if c and Path(c).is_file():
            return Path(c)
    return None


def yt_token_path() -> Path:
    return Path(os.environ.get("DF_YT_TOKEN") or "/content/youtube_token.json")


def yt_status() -> dict:
    return {"secret": yt_secret_path() is not None, "linked": yt_token_path().is_file()}


def yt_auth_link() -> str:
    sec = yt_secret_path()
    if sec is None:
        raise ValueError("client_secret.json မတွေ့ဘူး — Colab ထဲ /content/client_secret.json အဖြစ် တင်ပါ (Notebook Cell 5c)")
    try:
        from google_auth_oauthlib.flow import Flow
    except Exception:
        raise RuntimeError("google-auth-oauthlib မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ)")
    os.environ.setdefault("OAUTHLIB_RELAX_TOKEN_SCOPE", "1")
    try:
        flow = Flow.from_client_secrets_file(str(sec), scopes=YT_SCOPES, redirect_uri="http://localhost")
    except Exception as e:
        raise ValueError("client_secret.json ဖတ်မရဘူး (Google Cloud မှာ 'Desktop app' OAuth client ဖြစ်ရမယ်): " + str(e)[:120])
    url, _ = flow.authorization_url(access_type="offline", include_granted_scopes="true", prompt="consent")
    with _YT_LOCK:
        _YT["flow"] = flow
    return url


def yt_auth_complete(pasted: str) -> dict:
    with _YT_LOCK:
        flow = _YT["flow"]
    if flow is None:
        raise ValueError("Login link ကို အရင်ထုတ်ပါ (၁ ဆင့်)")
    code = sc.extract_oauth_code(pasted)
    if not code:
        raise ValueError("Google login ပြီးရင် browser မှာ ပေါ်တဲ့ http://localhost/?code=... လိပ်စာတစ်ခုလုံး ကူးထည့်ပါ")
    try:
        flow.fetch_token(code=code)
    except Exception as e:
        raise ValueError("Login အတည်မပြုနိုင်ဘူး: " + (sc.youtube_error_hint(str(e)) or str(e)[:160]))
    tp = yt_token_path()
    tp.write_text(flow.credentials.to_json(), encoding="utf-8")
    try:
        os.chmod(tp, 0o600)
    except OSError:
        pass
    with _YT_LOCK:
        _YT["flow"] = None
    return yt_status()


def yt_unlink():
    yt_token_path().unlink(missing_ok=True)
    with _YT_LOCK:
        _YT["flow"] = None


def _yt_service():
    try:
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from googleapiclient.discovery import build
    except Exception:
        raise RuntimeError("YouTube library မတပ်ရသေးဘူး (Colab Cell 1 ကို ပြန် run ပါ)")
    tp = yt_token_path()
    if not tp.is_file():
        raise ValueError("YouTube login မဝင်ရသေးဘူး — Studio > YouTube မှာ login အရင်ဝင်ပါ")
    creds = Credentials.from_authorized_user_file(str(tp), YT_SCOPES)
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
        tp.write_text(creds.to_json(), encoding="utf-8")
        try:
            os.chmod(tp, 0o600)
        except OSError:
            pass
    if not creds.valid:
        raise ValueError("YouTube login သက်တမ်းကုန်သွားပြီ — ပြန် login ဝင်ပါ")
    return build("youtube", "v3", credentials=creds)


def _srt_for(env: Env, video_name: str) -> Optional[Path]:
    stem = os.path.splitext(video_name)[0]
    for cand in (stem + "_my.srt", re.sub(r"_(cc|hardsub)$", "", stem) + "_my.srt", stem + ".srt"):
        if sc.safe_name(cand) and (env.lib / cand).is_file():
            return env.lib / cand
    return None


def op_youtube(env: Env, req) -> dict:
    if not getattr(req, "confirm_rights", False):
        raise ValueError("YouTube ပေါ် တင်မယ့် ဗီဒီယိုက ကိုယ်ပိုင် / တင်ခွင့်ရှိတာ ဖြစ်ကြောင်း အတည်ပြုပါ")
    names = list(getattr(req, "files", []) or [])
    if not names:
        raise ValueError("တင်မယ့် ဗီဒီယို မရွေးရသေးဘူး")
    paths = [file_path(env.lib, n, ("video",)) for n in names]
    privacy = getattr(req, "privacy", "private") or "private"
    if privacy not in sc.YT_PRIVACY:
        raise ValueError("privacy မမှန်ဘူး")
    thumb = None
    if getattr(req, "thumbnail", ""):
        thumb = file_path(env.lib, req.thumbnail, ("image",))
    youtube = _yt_service()
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload

    cat = sc.YT_CATEGORIES.get(getattr(req, "category", "entertainment"), "24")
    tags = sc.clean_tags(getattr(req, "tags", ""))
    desc = sc.clean_description(getattr(req, "description", ""))
    links, notes, errors = [], [], []
    for i, p in enumerate(paths, 1):
        env.check()
        title = sc.format_title(getattr(req, "title_template", "{name}"), os.path.splitext(p.name)[0], i, len(paths))
        body = {"snippet": {"title": title, "description": desc, "tags": tags, "categoryId": cat},
                "status": {"privacyStatus": privacy, "selfDeclaredMadeForKids": bool(getattr(req, "kids", False))}}
        try:
            media = MediaFileUpload(str(p), mimetype=MIME.get(p.suffix.lower(), "video/mp4"), chunksize=8 * 1024 * 1024, resumable=True)
            r = youtube.videos().insert(part="snippet,status", body=body, media_body=media)
            resp = None
            while resp is None:
                env.check()
                st, resp = r.next_chunk()
                if st:
                    env.progress(f"YouTube တင်နေတယ် ({i}/{len(paths)}) {int(st.progress() * 100)}%", int(100 * (i - 1 + st.progress()) / len(paths)))
            vid = resp.get("id")
            extra = ""
            if thumb is not None:
                try:
                    youtube.thumbnails().set(videoId=vid, media_body=MediaFileUpload(str(thumb))).execute()
                    extra += " · thumbnail OK"
                except Exception as te:
                    extra += " · thumbnail မတင်နိုင်ဘူး: " + str(te)[:80]
            if getattr(req, "upload_cc", False):
                srt = _srt_for(env, p.name)
                if srt is not None:
                    try:
                        youtube.captions().insert(
                            part="snippet", body={"snippet": {"videoId": vid, "language": "my", "name": "Myanmar", "isDraft": False}},
                            media_body=MediaFileUpload(str(srt), mimetype="application/octet-stream")).execute()
                        extra += " · Myanmar CC OK"
                    except Exception as ce:
                        extra += " · CC မတင်နိုင်ဘူး: " + str(ce)[:80]
                else:
                    extra += " · CC ဖိုင် (_my.srt) မတွေ့လို့ မတင်ဘူး"
            links.append({"title": title, "watch": f"https://youtu.be/{vid}", "studio": f"https://studio.youtube.com/video/{vid}/edit", "note": extra.strip(" ·")})
        except HttpError as he:
            msg = str(he)
            errors.append(f"{title}: " + (sc.youtube_error_hint(msg) or msg[:200]))
            if "quota" in msg.lower():
                break
        except Canceled:
            raise
        except Exception as e:
            msg = str(e)
            errors.append(f"{title}: " + (sc.youtube_error_hint(msg) or msg[:200]))
    if not links:
        raise RuntimeError(" | ".join(errors) or "တင်လို့မရဘူး")
    if privacy == "public":
        notes.append("Public ရွေးထားပေမဲ့ Google API project ကို audit မလုပ်ရသေးရင် YouTube က private ပဲ ထားတတ်တယ် — Studio လင့်ကနေ စစ်ပါ")
    notes.append("ဗီဒီယိုတစ်ပုဒ်တင်တိုင်း quota ကုန်တယ် (တစ်ရက် ၆ ပုဒ်လောက်)")
    env.progress("ပြီးပြီ", 100)
    return {"files": [], "links": links, "notes": notes, "errors": errors}

In [ ]:
# Cell 5b (ရွေးချယ်ခွင့် — YouTube က "robot မဟုတ်ကြောင်း အတည်ပြုပါ" လို့ တားမှသာ run ပါ)
# 1) Chrome မှာ YouTube ကို login ဝင်ထား၊ "Get cookies.txt LOCALLY" လို extension နဲ့ Netscape format cookies.txt ထုတ်ပါ။
# 2) ဒီ cell run ပြီး ထုတ်ထားတဲ့ ဖိုင်ကို ရွေးတင်ပါ။ ဖိုင်က သင့် Colab ထဲမှာပဲ နေမယ် (DramaFlow app ကို မဖြတ်ဘူး)။
# 3) ဒီ cookie က account login အတွက်ဖြစ်လို့ တခြားသူကို မပေးပါနဲ့၊ အဓိက account မဟုတ်တဲ့ အပိုတစ်ခုနဲ့ ထုတ်တာ ပိုစိတ်ချရတယ်။
# Server ပြေးနေပြီးသားဆိုရင်လည်း ဒီ cell ကို နောက်မှ run လို့ရတယ် — link တစ်ခုချင်းစီမှာ ဖိုင်ကို ပြန်ဖတ်တယ်။
from google.colab import files
import shutil
up = files.upload()
if up:
    shutil.move(next(iter(up)), "/content/cookies.txt")
    print("cookies.txt သိမ်းပြီး (/content/cookies.txt)")
else:
    print("ဘာမှ မတင်ဘူး — ကျော်လိုက်တယ်")

In [ ]:
# Cell 5c (ရွေးချယ်ခွင့် — Studio ကနေ YouTube ကို တင်ချင်မှသာ run ပါ)
# 1) console.cloud.google.com > project တစ်ခုဆောက် > "YouTube Data API v3" ကို Enable
# 2) OAuth consent screen (External) ဖြည့်၊ Test users ထဲ ကိုယ့် Gmail ထည့်
# 3) Credentials > Create credentials > OAuth client ID > Application type = "Desktop app" > JSON ဒေါင်းလုဒ်
# 4) ဒီ cell run ပြီး အဲဒီ JSON ကို ရွေးတင်ပါ။ ဖိုင်က သင့် Colab ထဲမှာပဲ နေမယ် (app ကို မဖြတ်ဘူး)။
# 5) App > Studio > YouTube မှာ "Login ဝင်" နှိပ်ပြီး ညွှန်ပြတဲ့အတိုင်း လုပ်ပါ။
# မှတ်ချက်: တစ်ရက်မှာ ၆ ပုဒ်လောက်ပဲ quota ရှိတယ်။ audit မလုပ်ရသေးတဲ့ API project က Public ပေးတာကို private ပဲ ထားတတ်တယ်။
from google.colab import files
import os, shutil
up = files.upload()
if up:
    shutil.move(next(iter(up)), "/content/client_secret.json")
    os.chmod("/content/client_secret.json", 0o600)
    print("client_secret.json သိမ်းပြီး (/content/client_secret.json)")
else:
    print("ဘာမှ မတင်ဘူး — ကျော်လိုက်တယ်")

In [ ]:
# Cell 5d (ရွေးချယ်ခွင့် — Studio ဖိုင်တွေ Colab ပိတ်ရင်လည်း မပျောက်ချင်မှသာ run ပါ — Server cell မတိုင်ခင်)
# Google Drive ထဲက "DramaFlow_Studio" folder ကို Studio library အဖြစ် သုံးမယ်။ ဗီဒီယိုကို ဒီ folder ထဲ တစ်ခါထည့်ထားရင်
# နောက်တစ်ခါ Colab စတိုင်း ပြန်မတင်ရတော့ဘူး၊ App > Render > "Colab မှာ ထုတ်မယ်" က ဒီဖိုင်ကို တိုက်ရိုက်သုံးမယ်။
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/DramaFlow_Studio", exist_ok=True)
os.environ["DF_STUDIO_DIR"] = "/content/drive/MyDrive/DramaFlow_Studio"
print("Studio library =", os.environ["DF_STUDIO_DIR"])

In [ ]:
# Cell 6: server စတင် + public URL ထုတ် (ဒီ cell က ဆက်လည်ပတ်နေရမယ် — မရပ်ပါနဲ့)
import os, re, secrets, subprocess, time

TOKEN = secrets.token_urlsafe(18)
env = dict(os.environ, APP_TOKEN=TOKEN)
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    cwd="/content/dramaflow_backend", env=env,
)
time.sleep(4)
if server.poll() is not None:
    raise SystemExit("Server မတက်ဘူး — အပေါ်က error ကို ကြည့်ပါ (Cell 1 run ပြီးလား)")

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Public URL ခဏစောင့်ပါ ...")
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nApp ရဲ့ Backend ကွက်မှာ ဒီတစ်ကြောင်းလုံး ကူးထည့်ပါ:")
        print(f"{m.group(0)}#{TOKEN}")
        break
for _ in proc.stdout:  # keep the cell (and the tunnel) alive
    pass